# K1 — Quality-gated anatomy routing on CardiacNet-PAH

Implementation of experiment **K1** from *Kaggle Experiment Designs* (Sep 29, 2026). One notebook holds the
whole stage chain; each Kaggle session runs one stage (or a fold subset of one) and every stage reads the previous
stage's output dataset and writes its own to `/kaggle/working`, which is then saved as a private dataset version.

**Research question.** In a ≤5 M-parameter model on the only public RHC-confirmed PH echo benchmark, does a learned
estimate of chamber-mask quality, used to gate how much anatomical evidence reaches the PAH head, improve
discrimination and calibration over an entropy gate and over joint training with gradient-conflict control, while
four-chamber Dice is preserved?

| Hypothesis | Test implemented here |
| --- | --- |
| H1 quality gate (G2) beats entropy gate (G1); gain vanishes under shuffled quality (G3) | 5-fold × 3-seed paired interval, group bootstrap, permutation test, Holm correction (stage 7) |
| H2 freezing the segmentation path keeps RV Dice within 0.02 of the dedicated segmenter (C4); joint + CAGrad (G5) does not close the gap without costing Dice | RV Dice of G5 vs C4 and AUROC of G5 vs G2 (stages 4, 6, 7) |
| H3 teacher-distilled init (D1) beats no-distillation init (D0) by ≥0.02 AUROC | D0 trains the final segmenter only (5 runs); H3 is tested on (a) C4 Dice and (b) the G1 entropy-gate arm under D0 vs D1, which needs no quality estimator. `K1_D0_FULL=1` adds the D0 inner cross-fit and the G2 comparison (stages 3–5, 7) |
| H4 frame-difference channel gain survives at matched parameters (A7 vs A0, A8) | screening arms at 5 folds × 1 seed, promoted to 3 seeds if the effect exceeds seed noise (stage 6) |

**Stage → session map** (set `K1_STAGES`, `K1_FOLDS`, `K1_SEEDS`, `K1_INITS` as environment variables or edit `CFG`).

| Stage | Session | Work | Output dataset |
| --- | --- | --- | --- |
| 0 | S0 (CPU) | label-id and time-axis audit: overlay panels per raw id, centroid statistics, suggested chamber mapping; stage 1 refuses to run until `CFG.cardiacnet_label_map` is set and `K1_LABEL_MAP_CONFIRMED=1` | `stage0/` in the working dir |
| 1 | S1 | audit, SHA-256 dedup, contradictory-label quarantine, acquisition groups (`branch/class_folder/token`, evidence printed, gate `K1_GROUPING_CONFIRMED=1`), one stratified group fold draw over both branches + manifest hash, 4 × 16-frame samplings per video at 112 px (student) and 224 px (teacher), sampling 0 annotation-preserving, flow fields | `cardiacnet-cache-v1` |
| 2 | S2 | frozen teacher latents over CardiacNet + CAMUS + HMC-QU clips, 4 temporal crops each | `teacher-latents-v1` |
| 3 | S3 | student encoder distillation (masked-latent L1 + cosine, EchoCLR-style same-study positives) | `student-init-v1` |
| 4 | S4–S5 | per fold: 3 inner cross-fit segmenters (out-of-sample masks for the quality estimator only) + final segmenter; HGB and MLP quality estimators on the OOS targets; diagnostic features for **every** labelled video of the fold from the **final** core; orientation audit | `seg-cores-v1`, `diag-features-v1` |
| 5 | S6 | G0–G4 × folds × seeds on cached features (adapters, gate, heads only) + D0 confirmatory subset | `diag-results-v1` |
| 6 | S7–S8 | G5 joint + CAGrad with the same fixed per-video q̂ as G2; A0/A7/A8 input ablations; C1 (original CaRDNet-X Lite import), C2 (Kinetics-initialised R3D-18); C4 from stage 4; seed-promotion rule | `joint-results-v1`, `ablation-results-v1` |
| 7 | S9 | ensembles, MSR / ensemble / MC-dropout / EDL, temperature vs Platt, conformal abstention, AUGRC, group bootstrap, permutation tests, Holm, latency/MACs, withdrawal conditions | `k1-final-v1` |

**Running without the data (validation mode).** When `/kaggle/input` is absent or `SMOKE_TEST=1`, the notebook
synthesises a small CardiacNet-like corpus (videos, four-chamber masks on five frames, PAH/ASD labels, acquisition
groups, deliberate duplicates and one contradictory label) and a frozen random teacher, and runs every stage end to
end at 64 px on CPU (default capacity 3.4 M parameters; the 0.9 M `small` variant is the capacity ablation). This validates the code paths, not the science; numbers from smoke mode are meaningless.

**Confirmation gates before the first real session.** Stage 0 produces the evidence for the chamber-id map and the time axis; stage 1 prints the acquisition-token evidence. Both gates (`K1_LABEL_MAP_CONFIRMED`, `K1_GROUPING_CONFIRMED`) must be set explicitly; they are bypassed only in smoke mode. Teacher weights load through `teacher_adapters/echojepa_l.py` (MIT) and `teacher_adapters/panecho.py` (CC BY-NC-SA); C1 imports the original CaRDNet-X Lite class from the `cardnet-x-lite-code` dataset; C2 needs `r3d_18-*.pth` (Kinetics-400) in the teacher-weights dataset.

**Confirmatory family (5 folds × 3 seeds):** G2 vs G1, G2 vs G3, G2 vs G0, D1 vs D0 (on G1). **Screening family (5 folds × 1 seed, promoted to 3 seeds if the effect exceeds seed noise):** G5, A0/A7/A8, C1, C2, the `small` capacity variant.


In [1]:
# ---------------------------------------------------------------- Cell 1: configuration and environment
import os, sys, io, re, json, time, math, glob, copy, pickle, hashlib, random, shutil, platform, warnings, itertools
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
torch.backends.cudnn.benchmark = True

ON_KAGGLE = Path("/kaggle/input").exists()
SMOKE_TEST = os.environ.get("SMOKE_TEST", "0" if ON_KAGGLE else "1") == "1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count()


def _env_list(name, default, cast=int):
    v = os.environ.get(name)
    return list(default) if v is None else [cast(x) for x in v.split(",") if x]


@dataclass
class K1Config:
    # ---- session control: which stages / folds / seeds / inits this session runs
    stages: List[int] = field(default_factory=lambda: _env_list("K1_STAGES", [0, 1, 2, 3, 4, 5, 6, 7]))
    folds: List[int] = field(default_factory=lambda: _env_list("K1_FOLDS", [0, 1, 2, 3, 4]))
    seeds: List[int] = field(default_factory=lambda: _env_list("K1_SEEDS", [0, 1, 2]))
    inits: List[str] = field(default_factory=lambda: _env_list("K1_INITS", ["D1", "D0"], str))
    # ---- paths (private datasets attached to the Kaggle notebook)
    kaggle_input: Path = Path("/kaggle/input")
    work: Path = Path("/kaggle/working") if ON_KAGGLE else Path("./k1_work")
    cardiacnet_root: Path = Path(os.environ.get("CARDIACNET_ROOT", "/kaggle/input/cardiacnet"))
    camus_root: Path = Path(os.environ.get("CAMUS_ROOT", "/kaggle/input/camus"))
    hmcqu_root: Path = Path(os.environ.get("HMCQU_ROOT", "/kaggle/input/hmc-qu"))
    teacher_weights: Path = Path(os.environ.get("TEACHER_WEIGHTS", "/kaggle/input/echojepa-weights"))
    teacher_name: str = os.environ.get("TEACHER_NAME", "echojepa_l")   # adapter module name in teacher_adapters/ (echojepa_l | panecho)
    cardnet_code_dir: Path = Path(os.environ.get("CARDNET_CODE_DIR", "/kaggle/input/cardnet-x-lite-code"))
    cardnet_import: str = os.environ.get("CARDNET_IMPORT", "cardnet_x_lite:CaRDNetXLite")   # module:Class of the original C1 definition
    # ---- confirmation gates (Stage 0 prints the evidence; set these to True only after reading it)
    label_map_confirmed: bool = os.environ.get("K1_LABEL_MAP_CONFIRMED", "0") == "1"
    grouping_confirmed: bool = os.environ.get("K1_GROUPING_CONFIRMED", "0") == "1"
    # ---- data
    n_folds: int = 5
    frames: int = 16
    size: int = 112
    group_regex: str = r"^(.*?)(?:[_\-][A-Za-z]?\d+)?$"   # acquisition token = stem minus a trailing counter; Stage 1 prints the evidence
    group_scope: str = os.environ.get("K1_GROUP_SCOPE", "folder")   # "folder": group = branch/class_folder/token (default); "token": merge across folders and branches
    flip_mode: str = os.environ.get("K1_FLIP_MODE", "none")          # "none" (default) or "swap": horizontal flip + LV<->RV, LA<->RA id swap
    capacity: str = os.environ.get("K1_CAPACITY", "default")          # "default" (3–4.5 M measured) or "small" (0.86 M capacity ablation)
    n_samplings: int = 4                                              # 16-frame samplings per video (0 = annotation-preserving, others = phase offsets for SSL)
    teacher_size: int = 224                                           # teacher input side stored in the cache
    class_names: Tuple[str, ...] = ("background", "LV", "RV", "LA", "RA")
    cardiacnet_label_map: Dict[int, int] = field(default_factory=lambda: {1: 1, 2: 2, 3: 3, 4: 4})  # raw id -> LV,RV,LA,RA  (VERIFY)
    cardiacnet_time_axis: int = -1   # axis of the NIfTI array that indexes frames (VERIFY)
    # ---- training
    batch: int = 8
    amp: bool = True
    num_workers: int = 2
    distill_epochs: int = 30
    seg_epochs: int = 40
    diag_epochs: int = 80
    joint_epochs: int = 40
    screen_epochs: int = 40
    lr_core: float = 3e-4
    lr_head: float = 3e-3
    teacher_crops: int = 4
    mask_ratio: float = 0.4
    inner_splits: int = 3
    val_frac: float = 0.20
    d0_full: bool = os.environ.get("K1_D0_FULL", "0") == "1"   # D0 inner cross-fit too (doubles stage-4 cost)
    mc_passes: int = 20
    n_boot: int = 2000
    n_perm: int = 2000
    target_sensitivity: float = 0.90
    coverage: float = 0.90
    # ---- chained artefact (dataset) names
    ds_cache: str = "cardiacnet-cache-v1"
    ds_latents: str = "teacher-latents-v1"
    ds_student: str = "student-init-v1"
    ds_cores: str = "seg-cores-v1"
    ds_feats: str = "diag-features-v1"
    ds_diag: str = "diag-results-v1"
    ds_joint: str = "joint-results-v1"
    ds_abl: str = "ablation-results-v1"
    ds_final: str = "k1-final-v1"


CFG = K1Config()
if SMOKE_TEST:  # small, CPU-sized settings that exercise every code path
    CFG.size, CFG.frames, CFG.batch, CFG.num_workers, CFG.teacher_size = 64, 16, 4, 0, 64
    CFG.distill_epochs, CFG.seg_epochs, CFG.diag_epochs, CFG.joint_epochs, CFG.screen_epochs = 2, 2, 30, 2, 1
    CFG.folds = _env_list("K1_FOLDS", [0, 1]); CFG.seeds = _env_list("K1_SEEDS", [0, 1, 2])
    CFG.n_boot, CFG.n_perm, CFG.mc_passes = 200, 200, 5
CFG.work.mkdir(parents=True, exist_ok=True)

CHAMBERS = ("LV", "RV", "LA", "RA")
CAP = {"default": dict(widths=(32, 64, 128, 256), blocks=(2, 3, 4, 3), expand=2, d_model=256, ff_mult=4, fpn_mid=96, fpn_convs=2),
       "small": dict(widths=(24, 48, 96, 192), blocks=(2, 2, 3, 2), expand=1, d_model=192, ff_mult=2, fpn_mid=48, fpn_convs=1)}[CFG.capacity]
D_FEAT = CAP["d_model"]
N_CLS = len(CFG.class_names)
IGNORE = 255


def code_hash() -> str:
    """Hash of the notebook source (Kaggle exposes it as __notebook_source__.ipynb); falls back to the config."""
    for p in ["__notebook_source__.ipynb"] + sorted(glob.glob("*.ipynb")):
        if Path(p).exists():
            return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:12]
    return hashlib.sha256(json.dumps(asdict(CFG), default=str, sort_keys=True).encode()).hexdigest()[:12]


ENV_INFO = dict(
    kaggle_image=os.environ.get("KAGGLE_DOCKER_IMAGE", os.environ.get("KAGGLE_CONTAINER_NAME", "local")),
    kaggle_run_type=os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "local"),
    python=platform.python_version(), torch=torch.__version__,
    device=str(DEVICE), gpu=(torch.cuda.get_device_name(0) if N_GPUS else "none"), n_gpus=N_GPUS,
    code_hash=code_hash(),
)


def stage_dir(name: str, create: bool = False) -> Path:
    """Resolve a chained artefact: this session's working copy first, then the attached input dataset."""
    local = CFG.work / name
    if local.exists() and any(local.iterdir()):
        return local
    attached = CFG.kaggle_input / name
    if attached.exists():
        return attached
    if create:
        local.mkdir(parents=True, exist_ok=True)
        return local
    raise FileNotFoundError(f"artefact '{name}' not found in {CFG.work} or {CFG.kaggle_input}; run the producing stage "
                            f"or attach the dataset version to this notebook")


STAGE_DEPS = {0: [], 1: [], 2: [CFG.ds_cache], 3: [CFG.ds_cache, CFG.ds_latents], 4: [CFG.ds_cache, CFG.ds_student],
              5: [CFG.ds_cache, CFG.ds_feats], 6: [CFG.ds_cache, CFG.ds_student, CFG.ds_cores, CFG.ds_feats, CFG.ds_diag],
              7: [CFG.ds_cache, CFG.ds_feats, CFG.ds_diag, CFG.ds_cores]}


def require_inputs(stage: int) -> None:
    """First-cell contract from the design doc: print the datasets a stage depends on; refuse to run if one is missing."""
    deps = STAGE_DEPS[stage]
    print(f"[stage {stage}] depends on: {deps or 'nothing'}")
    for d in deps:
        p = stage_dir(d)
        v = p / "VERSION.json"
        if not v.exists():
            raise RuntimeError(f"{d} at {p} has no VERSION.json — the producing stage did not finish")
        print(f"   {d:22s} -> {p}  ({json.loads(v.read_text()).get('created')})")


def write_version(name: str, extra: Optional[dict] = None) -> Path:
    p = stage_dir(name, create=True)
    (p / "VERSION.json").write_text(json.dumps({"name": name, "created": time.strftime("%Y-%m-%dT%H:%M:%S"),
                                                 "smoke": SMOKE_TEST, **ENV_INFO, **(extra or {})}, indent=1, default=str))
    return p


print(f"K1 | smoke={SMOKE_TEST} | device={DEVICE} ({ENV_INFO['gpu']}, n_gpus={N_GPUS}) | stages={CFG.stages} "
      f"folds={CFG.folds} seeds={CFG.seeds} inits={CFG.inits} | work={CFG.work.resolve()}")

K1 | smoke=True | device=cpu (none, n_gpus=0) | stages=[0, 1, 2, 3, 4, 5, 6, 7] folds=[0, 1] seeds=[0, 1, 2] inits=['D1', 'D0'] | work=/tmp/claude-0/-home-user-ph/d8985a5b-cc87-5db9-b4e3-f560f4f33009/scratchpad/run_k1/k1_work


In [2]:
# ---------------------------------------------------------------- Cell 2: utilities (seeding, logging, checkpoints, metrics)
from scipy import ndimage, stats
from scipy.spatial import ConvexHull
from scipy.optimize import minimize_scalar
from scipy.special import expit
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, GroupShuffleSplit


def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def peak_mem_mb() -> float:
    return torch.cuda.max_memory_allocated() / 2 ** 20 if torch.cuda.is_available() else float("nan")


def reset_peak_mem() -> None:
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def unwrap(m: nn.Module) -> nn.Module:
    return m.module if isinstance(m, nn.DataParallel) else m


def parallelise(m: nn.Module) -> nn.Module:
    """Use both T4s for the raw-video stages (DataParallel; ~1.8x throughput per the design doc)."""
    m = m.to(DEVICE)
    return nn.DataParallel(m) if N_GPUS > 1 else m


def count_params(m: nn.Module, trainable_only=False) -> int:
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable_only))


class RunLog:
    """One CSV per run under <dir>/runs/, plus read() that concatenates them (the statistics stage reads CSVs, not notebook output)."""
    def __init__(self, root: Path):
        self.dir = Path(root) / "runs"; self.dir.mkdir(parents=True, exist_ok=True)
    def append(self, run_id: str, row: dict) -> None:
        row = {"run_id": run_id, **row, **{f"env_{k}": v for k, v in ENV_INFO.items()},
               "smoke": SMOKE_TEST, "logged_at": time.strftime("%Y-%m-%dT%H:%M:%S")}
        pd.DataFrame([row]).to_csv(self.dir / f"{run_id}.csv", index=False)
    def read(self) -> pd.DataFrame:
        fs = sorted(self.dir.glob("*.csv"))
        return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()


class RunManifest:
    """Lists which fold-seed-arm combinations are complete so an unattended run picks the first missing one."""
    def __init__(self, path: Path):
        self.path = Path(path); self.done_set = set(json.loads(self.path.read_text())) if self.path.exists() else set()
    def done(self, rid: str) -> bool:
        return rid in self.done_set
    def mark(self, rid: str) -> None:
        self.done_set.add(rid); self.path.parent.mkdir(parents=True, exist_ok=True)
        self.path.write_text(json.dumps(sorted(self.done_set), indent=0))


def save_ckpt(path: Path, model, opt=None, sched=None, epoch=0, extra=None) -> None:
    """Checkpoint with model, optimizer, scheduler, epoch and RNG state (resume loses at most one epoch)."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model": unwrap(model).state_dict(), "opt": opt.state_dict() if opt else None,
                "sched": sched.state_dict() if sched else None, "epoch": epoch,
                "rng": {"torch": torch.get_rng_state(), "np": np.random.get_state(), "py": random.getstate(),
                        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
                "extra": extra or {}}, str(path) + ".tmp")
    os.replace(str(path) + ".tmp", path)


def load_ckpt(path: Path, model, opt=None, sched=None):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    unwrap(model).load_state_dict(ck["model"])
    if opt is not None and ck.get("opt"):
        opt.load_state_dict(ck["opt"])
    if sched is not None and ck.get("sched"):
        sched.load_state_dict(ck["sched"])
    r = ck.get("rng", {})
    if r:
        torch.set_rng_state(r["torch"]); np.random.set_state(r["np"]); random.setstate(r["py"])
        if r.get("cuda") is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(r["cuda"])
    return ck["epoch"], ck.get("extra", {})


def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(CFG.amp and DEVICE.type == "cuda"))


def make_scaler():
    return torch.amp.GradScaler("cuda", enabled=(CFG.amp and DEVICE.type == "cuda"))


# ---------------- classification metrics -----------------------------------------------------------------------
def _safe_auc(fn, y, p):
    y = np.asarray(y); p = np.asarray(p)
    return float(fn(y, p)) if len(np.unique(y)) == 2 else float("nan")


def ece_equal_mass(y, p, n_bins=15) -> float:
    y, p = np.asarray(y, float), np.asarray(p, float)
    if len(y) < n_bins:
        n_bins = max(1, len(y) // 2)
    order = np.argsort(p); bins = np.array_split(order, n_bins)
    return float(sum(len(b) * abs(y[b].mean() - p[b].mean()) for b in bins if len(b)) / max(1, len(y)))


def calibration_slope(y, p) -> float:
    """Slope of logit(y) ~ a + b·logit(p) fitted by logistic regression (1.0 = perfect)."""
    from sklearn.linear_model import LogisticRegression
    y = np.asarray(y); z = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))[:, None]
    if len(np.unique(y)) < 2:
        return float("nan")
    return float(LogisticRegression(C=1e6, max_iter=1000).fit(z, y).coef_[0, 0])


def binary_metrics(y, p) -> dict:
    y = np.asarray(y, float); p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    return dict(auroc=_safe_auc(roc_auc_score, y, p), auprc=_safe_auc(average_precision_score, y, p),
                brier=float(np.mean((p - y) ** 2)), nll=float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))),
                ece=ece_equal_mass(y, p), cal_slope=calibration_slope(y, p), acc=float(np.mean((p > 0.5) == y)), n=int(len(y)))


# ---------------- segmentation metrics -------------------------------------------------------------------------
def dice_binary(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool); s = a.sum() + b.sum()
    return 1.0 if s == 0 else float(2.0 * np.logical_and(a, b).sum() / s)


def _boundary(m: np.ndarray) -> np.ndarray:
    m = m.astype(bool)
    return m & ~ndimage.binary_erosion(m)


def surface_distances(pred: np.ndarray, gt: np.ndarray) -> Tuple[float, float]:
    """(mean symmetric surface distance, HD95 = 95th percentile of the symmetric surface distances), both in PIXELS (no trustworthy
    spacing in CardiacNet); nan when either mask is empty."""
    bp, bg = _boundary(pred), _boundary(gt)
    if bp.sum() == 0 or bg.sum() == 0:
        return float("nan"), float("nan")
    dt_g = ndimage.distance_transform_edt(~bg); dt_p = ndimage.distance_transform_edt(~bp)
    d = np.concatenate([dt_g[bp], dt_p[bg]])
    return float(d.mean()), float(np.percentile(d, 95))


def holm(pvals: List[float]) -> List[float]:
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p); adj = np.empty(m)
    run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj.tolist()


class Timer:
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *a): self.dt = time.time() - self.t

In [3]:
# ---------------------------------------------------------------- Cell 3: data — records, synthetic corpus, real loaders, cached dataset
# A "record" is a dict: id, source ('PAH','ASD','CAMUS','HMCQU'), path, label (0/1 or None), group, sha256,
# frames (uint8 [T,H,W], loaded lazily via record['load']()), masks ({frame_idx: uint8 [H,W] with unified ids}).

def token_of(stem: str) -> str:
    m = re.match(CFG.group_regex, stem)
    return m.group(1) if m and m.group(1) else stem


def make_group(branch: str, class_folder: str, token: str) -> str:
    """Acquisition group key. 'folder' (default): tokens are local to (branch, class folder) as the earlier CaRDNet-X audit found;
    'token': conservative sensitivity setting that merges the same token across folders and branches."""
    return f"{branch}/{class_folder}/{token}" if CFG.group_scope == "folder" else token


FLIP_SWAP = np.array([0, 2, 1, 4, 3], np.uint8)     # LV<->RV, LA<->RA when the image is mirrored (CFG.flip_mode == "swap")


# ---------------- synthetic CardiacNet-like corpus (validation mode only) ---------------------------------------
def _ellipse(H, W, cy, cx, ry, rx):
    yy, xx = np.mgrid[0:H, 0:W]
    return (((yy - cy) / max(ry, 1e-3)) ** 2 + ((xx - cx) / max(rx, 1e-3)) ** 2) <= 1.0


def synth_video(rng, T=20, H=96, W=96, pah=0, style=None, n_annot=5):
    """Apical-4-chamber cartoon: four cavities (dark) in bright walls inside an echo sector, beating over T frames.
    PAH videos have a larger RV / smaller LV so that RV/LV area ratio carries signal, as in the real task."""
    style = style or dict(gain=rng.uniform(0.7, 1.3), offset=rng.uniform(-0.1, 0.1), speckle=rng.uniform(0.05, 0.2), shift=rng.integers(-4, 5, 2))
    rv_scale = 0.75 + 0.45 * pah + rng.normal(0, 0.10)
    lv_scale = 1.05 - 0.15 * pah + rng.normal(0, 0.08)
    phase, cycles = rng.uniform(0, 2 * np.pi), rng.uniform(0.8, 1.5)
    cy0, cx0 = H * 0.5 + style["shift"][0], W * 0.5 + style["shift"][1]
    sector = _ellipse(H, W, 0, cx0, H * 1.05, W * 0.62) & (np.mgrid[0:H, 0:W][0] > 2)
    yy, xx = np.mgrid[0:H, 0:W]
    frames = np.zeros((T, H, W), np.uint8); masks = {}
    annot = np.linspace(0, T - 1, n_annot).round().astype(int)
    for t in range(T):
        con = 0.5 + 0.5 * np.cos(2 * np.pi * cycles * t / T + phase)   # 1 = end-diastole
        v = 0.82 + 0.18 * con; a = 1.0 - 0.15 * con
        lv = _ellipse(H, W, cy0 + 0.13 * H, cx0 + 0.18 * W, 0.24 * H * v * lv_scale, 0.13 * W * v * lv_scale)
        rv = _ellipse(H, W, cy0 + 0.11 * H, cx0 - 0.16 * W, 0.22 * H * v * rv_scale, 0.12 * W * v * rv_scale)
        la = _ellipse(H, W, cy0 - 0.20 * H, cx0 + 0.16 * W, 0.11 * H * a, 0.11 * W * a)
        ra = _ellipse(H, W, cy0 - 0.20 * H, cx0 - 0.15 * W, 0.10 * H * a * (0.9 + 0.3 * pah), 0.10 * W * a)
        lab = np.zeros((H, W), np.uint8); lab[lv] = 1; lab[rv] = 2; lab[la] = 3; lab[ra] = 4
        wall = ndimage.binary_dilation(lab > 0, iterations=4) & (lab == 0)
        img = 0.25 + 0.55 * wall - 0.15 * (lab > 0) + 0.05 * np.sin(yy / 6.0 + t) * (lab == 0)
        img = img * style["gain"] + style["offset"] + rng.gamma(4.0, style["speckle"] / 4.0, (H, W)) - style["speckle"]
        img = ndimage.gaussian_filter(img, 0.8) * sector
        frames[t] = (np.clip(img, 0, 1) * 255).astype(np.uint8)
        if t in annot:
            masks[int(t)] = lab
    return frames, masks


def synth_records(n_pah=48, n_asd=16, n_extra=8, seed=0) -> List[dict]:
    rng = np.random.default_rng(seed); recs = []; g = 0
    def make(source, n, label_fn):
        nonlocal g
        i = 0
        while i < n:
            g += 1; style = dict(gain=rng.uniform(0.7, 1.3), offset=rng.uniform(-0.1, 0.1), speckle=rng.uniform(0.05, 0.2), shift=rng.integers(-4, 5, 2))
            for k in range(int(rng.integers(1, 4))):          # 1–3 videos per acquisition group share a style
                if i >= n: break
                y = label_fn()
                fr, mk = synth_video(rng, pah=y if source != "CAMUS" and source != "HMCQU" else 0, style=style)
                cf = ("" if source not in ("PAH", "ASD") else (source if y else f"Non-{source}")); tok = f"acq{g:03d}"
                recs.append(dict(id=f"{source}_acq{g:03d}_{k}", source=source, branch=source, class_folder=cf, token=tok, path=f"synthetic/{source}/{cf}/acq{g:03d}_{k}.nii.gz",
                                 label=(y if source in ("PAH", "ASD") else None), group=make_group(source, cf, tok), _frames=fr,
                                 masks=(mk if source in ("PAH", "ASD") else {})))
                i += 1
    make("PAH", n_pah, lambda: int(rng.random() < 0.65)); make("ASD", n_asd, lambda: int(rng.random() < 0.5))
    make("CAMUS", n_extra // 2, lambda: 0); make("HMCQU", n_extra - n_extra // 2, lambda: 0)
    # deliberate audit cases: an exact duplicate (kept once) and a duplicate with a contradictory label (quarantined)
    d = copy.deepcopy(recs[3]); d["id"] += "_dup"; d["path"] += ".dup"; recs.append(d)
    c = copy.deepcopy(recs[5]); c["id"] += "_contra"; c["path"] += ".contra"; c["label"] = 1 - c["label"]; recs.append(c)
    for r in recs:
        fr = r.pop("_frames"); r["load"] = (lambda fr=fr: fr); r["sha256"] = hashlib.sha256(fr.tobytes()).hexdigest()
        r["load_raw_masks"] = (lambda m=r["masks"]: m)     # synthetic raw ids == unified ids
        r["raw_shape"] = fr.shape
    return recs


# ---------------- real loaders (Kaggle) --------------------------------------------------------------------------
def _to_uint8(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.float32); lo, hi = np.percentile(a, [0.5, 99.5])
    return (np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)


def _read_volume(path: Path) -> np.ndarray:
    """NIfTI / MetaImage / video reader returning a float array with frames on axis 0 (uses CFG.cardiacnet_time_axis for NIfTI)."""
    s = str(path)
    if s.endswith((".nii", ".nii.gz")):
        import nibabel as nib
        a = np.asanyarray(nib.load(s).dataobj)
        if a.ndim == 4: a = a[..., 0] if a.shape[-1] == 1 else a.squeeze()
        return np.moveaxis(a, CFG.cardiacnet_time_axis, 0) if a.ndim == 3 else a[None]
    if s.endswith(".mhd"):
        try:
            import SimpleITK as sitk
            return sitk.GetArrayFromImage(sitk.ReadImage(s)).astype(np.float32)
        except ImportError:
            hdr = dict(l.split("=", 1) for l in Path(s).read_text().splitlines() if "=" in l)
            hdr = {k.strip(): v.strip() for k, v in hdr.items()}
            dims = [int(x) for x in hdr["DimSize"].split()][::-1]
            dt = {"MET_UCHAR": np.uint8, "MET_SHORT": np.int16, "MET_USHORT": np.uint16, "MET_FLOAT": np.float32, "MET_DOUBLE": np.float64}[hdr["ElementType"]]
            raw = np.fromfile(Path(s).with_name(hdr["ElementDataFile"]), dtype=dt).reshape(dims)
            return raw.astype(np.float32) if raw.ndim == 3 else raw[None].astype(np.float32)
    if s.endswith((".avi", ".mp4")):
        import cv2
        cap, fr = cv2.VideoCapture(s), []
        while True:
            ok, f = cap.read()
            if not ok: break
            fr.append(cv2.cvtColor(f, cv2.COLOR_BGR2GRAY))
        cap.release(); return np.stack(fr).astype(np.float32)
    raise ValueError(f"unsupported file {s}")


def iter_cardiacnet(root: Path) -> List[dict]:
    """CardiacNet-PAH / -ASD Kaggle release. Expected: <root>/**/CardiacNet-{PAH,ASD}/{PAH,Non-PAH|ASD,Non-ASD}/<case>/(image|label).nii.gz
    or <case>.nii.gz + <case>_label.nii.gz side by side. VERIFY the layout, the label ids and the time axis on the listing."""
    recs = []
    for img in sorted(p for p in Path(root).rglob("*.nii*") if "label" not in p.name.lower() and "gt" not in p.name.lower()):
        cands = [q for q in img.parent.glob("*.nii*") if ("label" in q.name.lower() or "gt" in q.name.lower()) and (q.stem.split(".")[0].replace("_label", "").replace("label", "") in (img.stem.split(".")[0], "") or img.name.startswith("image"))]
        lab = cands[0] if cands else None
        parts = [p.lower() for p in img.parts]
        source = "ASD" if any("asd" in p for p in parts) else "PAH"
        y = None
        for p in parts:
            if re.fullmatch(r"non[-_ ]?(pah|asd)", p): y = 0
            elif re.fullmatch(r"(pah|asd)", p): y = 1
        if y is None: continue
        stem = img.parent.name if img.stem.startswith("image") else img.stem.split(".")[0]
        class_folder = next((p for p in img.parts if re.fullmatch(r"(non[-_ ]?)?(pah|asd)", p.lower())), "unknown")
        def load(img=img):
            return _to_uint8(_read_volume(img))
        def load_raw_masks(lab=lab):
            if lab is None: return {}
            v = _read_volume(lab).astype(np.int64)
            return {t: v[t].astype(np.uint8) for t in range(v.shape[0]) if v[t].any()}
        def load_masks(lab=lab):
            out = {}
            for t, raw in load_raw_masks(lab).items():
                m = np.zeros(raw.shape, np.uint8)
                for r_, uid in CFG.cardiacnet_label_map.items(): m[raw == r_] = uid
                out[t] = m
            return out
        tok = token_of(stem)
        recs.append(dict(id=f"{source}_{stem}", source=source, branch=source, class_folder=class_folder, token=tok, path=str(img), label=int(y),
                         group=make_group(source, class_folder, tok), sha256=hashlib.sha256(img.read_bytes()).hexdigest(), load=load, load_masks=load_masks,
                         load_raw_masks=load_raw_masks, raw_shape=None, label_path=(str(lab) if lab else None)))
    return recs


def iter_extras(root: Path, source: str) -> List[dict]:
    """Unlabelled extras for distillation: CAMUS half-cycle sequences (*_sequence.nii.gz|.mhd) or HMC-QU A4C videos (*.avi)."""
    pats = ["**/*_sequence.nii.gz", "**/*_sequence.mhd"] if source == "CAMUS" else ["**/*.avi"]
    recs = []
    for pat in pats:
        for f in sorted(Path(root).glob(pat)):
            stem = f.stem.split(".")[0]
            recs.append(dict(id=f"{source}_{stem}", source=source, branch=source, class_folder="", token=token_of(stem), path=str(f), label=None, group=make_group(source, "", token_of(stem)),
                             sha256=hashlib.sha256(f.read_bytes()).hexdigest(), load=(lambda f=f: _to_uint8(_read_volume(f))), load_masks=lambda: {}, load_raw_masks=lambda: {}, raw_shape=None))
    return recs


def discover_records() -> List[dict]:
    if SMOKE_TEST:
        return synth_records()
    recs = iter_cardiacnet(CFG.cardiacnet_root)
    if not recs:
        raise FileNotFoundError(f"no CardiacNet volumes under {CFG.cardiacnet_root}")
    for root, src in ((CFG.camus_root, "CAMUS"), (CFG.hmcqu_root, "HMCQU")):
        if Path(root).exists():
            recs += iter_extras(root, src)
        else:
            print(f"[data] {src} not attached ({root}); distillation runs without it")
    return recs


# ---------------- frame sampling and resizing --------------------------------------------------------------------
def annotation_preserving_indices(T: int, n: int, annotated: List[int]) -> np.ndarray:
    """Uniform n indices over [0,T-1], with the nearest uniform slots replaced so that every annotated frame is included."""
    idx = np.linspace(0, T - 1, n).round().astype(int)
    for a in sorted(set(int(x) for x in annotated))[:n]:
        if a in idx: continue
        free = [i for i in range(n) if idx[i] not in annotated]
        j = min(free, key=lambda i: abs(idx[i] - a)); idx[j] = a
    return np.sort(idx)


def phase_offset_indices(T: int, n: int, k: int, K: int) -> np.ndarray:
    """Sampling k of K: the uniform grid over the full source video shifted by k/K of one stride (SSL samplings 1..K-1)."""
    base = np.linspace(0, T - 1, n); stride = (T - 1) / max(n - 1, 1)
    return np.clip(np.round(base + k * stride / K), 0, T - 1).astype(int)


def resize_stack(x: np.ndarray, size: int, nearest=False) -> np.ndarray:
    t = torch.from_numpy(np.ascontiguousarray(x)).float()[:, None]
    y = F.interpolate(t, size=(size, size), mode="nearest" if nearest else "bilinear", **({} if nearest else {"align_corners": False}))
    return y[:, 0].numpy()


class ClipDataset(torch.utils.data.Dataset):
    """Reads sampling 0 of the cache (student resolution). Returns x [C,T,H,W] (grayscale + |I_t − I_{t−2}| by default), mask [T,H,W]
    (255 = unannotated), y [2] (nan = unknown). Augmentation follows CFG.flip_mode: 'none' (default) or 'swap' (mirror + LV<->RV, LA<->RA)."""
    def __init__(self, index: pd.DataFrame, cache: Path, norm: dict, channels="diff", flow_dir: Optional[Path] = None, augment=False):
        self.df = index.reset_index(drop=True); self.cache = Path(cache); self.norm = norm; self.channels = channels; self.flow_dir = flow_dir; self.augment = augment
    def __len__(self): return len(self.df)
    @staticmethod
    def frame_difference(g: torch.Tensor) -> torch.Tensor:     # g [T,H,W] in [0,1]
        d = torch.zeros_like(g); d[2:] = (g[2:] - g[:-2]).abs(); return d
    def __getitem__(self, i):
        r = self.df.iloc[i]; z = np.load(self.cache / "clips" / f"{r.id}.npz")
        g = torch.from_numpy(z["frames"]).float() / 255.0; m = z["masks"]
        if self.augment:
            if CFG.flip_mode == "swap" and random.random() < 0.5:
                g = g.flip(-1); m = np.where(m == IGNORE, IGNORE, FLIP_SWAP[np.minimum(m, 4)])[..., ::-1].copy()
            g = (g * random.uniform(0.85, 1.15) + random.uniform(-0.05, 0.05)).clamp(0, 1)
        m = torch.from_numpy(np.ascontiguousarray(m)).long()
        gi = (g - self.norm["mean"]) / self.norm["std"]
        if self.channels == "diff":
            x = torch.stack([gi, self.frame_difference(g) / self.norm["diff_std"]])
        elif self.channels == "intensity":                        # A7: intensity only, second channel zero (parameters identical)
            x = torch.stack([gi, torch.zeros_like(gi)])
        elif self.channels == "raw2":                             # A8: raw intensity twice; the model learns its own temporal channel
            x = torch.stack([gi, g])
        elif self.channels == "flow":
            fl = torch.from_numpy(np.load(self.flow_dir / f"{r.id}.npy").astype(np.float32))
            x = torch.stack([gi, fl / max(self.norm.get("flow_std", 1.0), 1e-6)])
        else:
            raise ValueError(self.channels)
        y = torch.tensor([r.y_pah, r.y_asd], dtype=torch.float32)
        return x, m, y, i

In [4]:
# ---------------------------------------------------------------- Cell 4: models (student core, decoder, heads, baselines)
class DWSep2p1D(nn.Module):
    """Depthwise-separable (2+1)D residual block: spatial DW 1×3×3 → PW expansion (×expand) → temporal DW 3×1×1 → PW projection; BatchNorm;
    spatial-only stride."""
    def __init__(self, cin, cout, stride=1, expand=CAP["expand"]):
        super().__init__(); mid = cout * expand
        self.s_dw = nn.Conv3d(cin, cin, (1, 3, 3), stride=(1, stride, stride), padding=(0, 1, 1), groups=cin, bias=False)
        self.s_pw = nn.Conv3d(cin, mid, 1, bias=False); self.bn1 = nn.BatchNorm3d(mid)
        self.t_dw = nn.Conv3d(mid, mid, (3, 1, 1), padding=(1, 0, 0), groups=mid, bias=False)
        self.t_pw = nn.Conv3d(mid, cout, 1, bias=False); self.bn2 = nn.BatchNorm3d(cout)
        self.short = None if (cin == cout and stride == 1) else nn.Sequential(nn.Conv3d(cin, cout, 1, stride=(1, stride, stride), bias=False), nn.BatchNorm3d(cout))
    def forward(self, x):
        idt = x if self.short is None else self.short(x)
        y = F.relu(self.bn1(self.s_pw(self.s_dw(x))))
        return F.relu(self.bn2(self.t_pw(self.t_dw(y))) + idt)


class StudentEncoder(nn.Module):
    """4 stages of depthwise-separable R(2+1)D blocks (widths/blocks from CFG.capacity); spatial-only pooling; returns the 4 pyramid levels (1/2 … 1/16)."""
    def __init__(self, in_ch=2, widths=CAP["widths"], blocks=CAP["blocks"]):
        super().__init__(); self.widths = widths
        self.stem = nn.Sequential(nn.Conv3d(in_ch, widths[0], (1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1), bias=False), nn.BatchNorm3d(widths[0]), nn.ReLU(inplace=True))
        stages, cin = [], widths[0]
        for i, (w, n) in enumerate(zip(widths, blocks)):
            layers = [DWSep2p1D(cin, w, stride=1 if i == 0 else 2)] + [DWSep2p1D(w, w) for _ in range(n - 1)]
            stages.append(nn.Sequential(*layers)); cin = w
        self.stages = nn.ModuleList(stages)
    def forward(self, x):
        x = self.stem(x); feats = []
        for s in self.stages:
            x = s(x); feats.append(x)
        return feats


class TemporalTransformer(nn.Module):
    """2-layer, 4-head transformer over the bottleneck tokens (one token per frame, spatially pooled)."""
    def __init__(self, d=D_FEAT, heads=4, layers=2, max_t=64, p=0.1, ff_mult=CAP["ff_mult"]):
        super().__init__()
        self.pos = nn.Parameter(torch.zeros(1, max_t, d)); nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(d, heads, ff_mult * d, dropout=p, batch_first=True, norm_first=True, activation="gelu")
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False); self.norm = nn.LayerNorm(d)
    def forward(self, f4):                                     # [B,C,T,h,w] -> tokens [B,T,C]
        tok = f4.mean(dim=(3, 4)).transpose(1, 2)
        return self.norm(self.enc(tok + self.pos[:, :tok.shape[1]]))


class LightFPN(nn.Module):
    """Light FPN with deep supervision (a head at every pyramid level); frames are folded into the batch (2-D decoding).
    `n_convs` full 3×3 smoothing convolutions per level (CFG.capacity)."""
    def __init__(self, in_chs=CAP["widths"], mid=CAP["fpn_mid"], n_cls=N_CLS, n_convs=CAP["fpn_convs"]):
        super().__init__()
        self.lat = nn.ModuleList(nn.Conv2d(c, mid, 1) for c in in_chs)
        def smooth():
            return nn.Sequential(*[m for _ in range(n_convs) for m in (nn.Conv2d(mid, mid, 3, padding=1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(inplace=True))])
        self.smooth = nn.ModuleList(smooth() for _ in in_chs)
        self.heads = nn.ModuleList(nn.Conv2d(mid, n_cls, 1) for _ in in_chs)
    def forward(self, feats, out_hw):
        B, _, T, _, _ = feats[0].shape
        f2d = [f.transpose(1, 2).flatten(0, 1) for f in feats]
        x = self.smooth[3](self.lat[3](f2d[3])); outs = [self.heads[3](x)]
        for i in (2, 1, 0):
            x = F.interpolate(x, size=f2d[i].shape[-2:], mode="bilinear", align_corners=False) + self.lat[i](f2d[i])
            x = self.smooth[i](x); outs.append(self.heads[i](x))
        outs = [F.interpolate(o, size=out_hw, mode="bilinear", align_corners=False) for o in outs]
        return [o.view(B, T, *o.shape[1:]).transpose(1, 2) for o in outs]      # coarse → fine; the last one is the prediction


class SegModel(nn.Module):
    """Encoder + temporal module + FPN decoder. forward → (list of deep-supervised logits [B,5,T,H,W], enriched f4, tokens)."""
    def __init__(self, in_ch=2, n_cls=N_CLS):
        super().__init__()
        self.encoder = StudentEncoder(in_ch); self.temporal = TemporalTransformer(); self.decoder = LightFPN(n_cls=n_cls)
    def core_parameters(self):
        return list(self.encoder.parameters()) + list(self.temporal.parameters())
    def forward_features(self, x):
        feats = self.encoder(x); tok = self.temporal(feats[-1])
        feats[-1] = feats[-1] + tok.transpose(1, 2)[..., None, None]
        return feats, tok
    def forward(self, x):
        feats, tok = self.forward_features(x)
        return self.decoder(feats, x.shape[-2:]), feats[-1], tok


def seg_loss(logits_list, mask):
    """Deep-supervised CE (ignore 255) + soft Dice over LV/RV/LA/RA on annotated frames only."""
    valid = mask != IGNORE
    if valid.sum() == 0:
        return logits_list[-1].sum() * 0.0
    loss, weights = 0.0, [0.125, 0.25, 0.5, 1.0]
    for w, lg in zip(weights, logits_list):
        ce = F.cross_entropy(lg.float(), mask, ignore_index=IGNORE)
        p = lg.float().softmax(1).permute(0, 2, 3, 4, 1)[valid]     # [Nvalid, C]
        g = F.one_hot(mask[valid], N_CLS).float()
        dice = (2 * (p * g).sum(0)[1:] + 1) / (p.sum(0)[1:] + g.sum(0)[1:] + 1)
        loss = loss + w * (ce + 1 - dice.mean())
    return loss / sum(weights)


class LearnedTemporalBlock(nn.Module):
    """A8: replaces the hand-crafted |I_t − I_{t−2}| channel with a learned 5-tap temporal filter (6 parameters), initialised to it."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv3d(1, 1, (5, 1, 1), padding=(2, 0, 0))
        with torch.no_grad():
            self.conv.weight.zero_(); self.conv.weight[0, 0, 0] = -1.0; self.conv.weight[0, 0, 2] = 1.0; self.conv.bias.zero_()
    def forward(self, x):                                        # x [B,2,T,H,W] with raw intensity in channel 1
        return torch.cat([x[:, :1], self.conv(x[:, 1:2]).abs()], 1)


class ClipClassifier(nn.Module):
    """End-to-end classifier on the student core (A0/A7/A8 and the D0/D1 stems): pooled tokens → 2 logits (PAH, ASD)."""
    def __init__(self, core: Optional[SegModel] = None, temporal_block=False, p=0.3):
        super().__init__()
        self.core = core or SegModel(); self.pre = LearnedTemporalBlock() if temporal_block else nn.Identity()
        self.head = nn.Sequential(nn.Dropout(p), nn.Linear(D_FEAT, 64), nn.GELU(), nn.Dropout(p), nn.Linear(64, 2))
    def forward(self, x):
        _, tok = self.core.forward_features(self.pre(x)); return self.head(tok.mean(1))


class _CaRDNetXLiteStandIn(nn.Module):
    """Size-matched stand-in used ONLY in validation mode. The real C1 is imported from the user's CaRDNet-X Lite code dataset (B2)."""
    def __init__(self, in_ch=2, widths=(32, 64, 128, 256)):
        super().__init__()
        def blk(ci, co, s):
            return nn.Sequential(nn.Conv3d(ci, co, (1, 3, 3), stride=(1, s, s), padding=(0, 1, 1), bias=False), nn.BatchNorm3d(co), nn.ReLU(inplace=True),
                                 nn.Conv3d(co, co, (3, 1, 1), padding=(1, 0, 0), bias=False), nn.BatchNorm3d(co), nn.ReLU(inplace=True))
        layers, ci = [], in_ch
        for i, w in enumerate(widths):
            layers += [blk(ci, w, 1 if i == 0 else 2), blk(w, w, 1)]; ci = w
        self.body = nn.Sequential(*layers); self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(ci, 2))
    def forward(self, x):
        return self.head(self.body(x).mean(dim=(2, 3, 4)))


CARDNET_X_LITE_PARAMS = 2_760_000      # the user's measured figure; the imported class must match within ±2 %


def build_cardnet_x_lite() -> nn.Module:
    """C1: the original CaRDNet-X Lite, imported from the `cardnet-x-lite-code` Kaggle dataset (CFG.cardnet_import = 'module:Class').
    Falls back to the stand-in only in validation mode."""
    mod_name, cls_name = CFG.cardnet_import.split(":")
    if str(CFG.cardnet_code_dir) not in sys.path: sys.path.insert(0, str(CFG.cardnet_code_dir))
    try:
        import importlib; cls = getattr(importlib.import_module(mod_name), cls_name); m = cls()
    except Exception as e:
        if SMOKE_TEST:
            return _CaRDNetXLiteStandIn()
        raise ImportError(f"cannot import {CFG.cardnet_import} from {CFG.cardnet_code_dir}: {e}; attach the CaRDNet-X Lite code dataset") from e
    n = count_params(m); assert abs(n - CARDNET_X_LITE_PARAMS) / CARDNET_X_LITE_PARAMS <= 0.02, f"C1 parameter count {n} differs from 2.76 M by more than 2 %"
    return m


def r3d18_two_channel() -> nn.Module:
    """C2 — torchvision 3D ResNet-18 initialised from Kinetics-400 (`r3d_18-*.pth` in the teacher-weights dataset) with the stem inflated
    to two channels. Random init is allowed only in validation mode."""
    import torchvision
    m = torchvision.models.video.r3d_18(weights=None); ck = next(iter(Path(CFG.teacher_weights).glob("r3d_18*.pth")), None) if Path(CFG.teacher_weights).exists() else None
    if ck is not None:
        sd = torch.load(ck, map_location="cpu", weights_only=False); m.load_state_dict(sd.get("state_dict", sd)); pretrained = True
    elif SMOKE_TEST:
        pretrained = False
    else:
        raise FileNotFoundError(f"Kinetics-400 checkpoint r3d_18-*.pth not found under {CFG.teacher_weights} (C2 must be Kinetics-initialised)")
    old = m.stem[0]; new = nn.Conv3d(2, old.out_channels, old.kernel_size, old.stride, old.padding, bias=False)
    with torch.no_grad():
        if pretrained: new.weight.copy_(old.weight.mean(1, keepdim=True).repeat(1, 2, 1, 1, 1) * 1.5)
        else: nn.init.kaiming_normal_(new.weight)
    m.stem[0] = new; m.fc = nn.Linear(m.fc.in_features, 2); m.kinetics_init = pretrained
    return m


class DiagNet(nn.Module):
    """Diagnostic model on cached, detached features.
    G0: global-context adapter only. G1/G2/G3/G4: chamber tokens scaled by gate_c · sigmoid(relevance[task,c]) where gate_c is
    1−entropy (G1), estimated quality q̂ (G2), shuffled q̂ (G3) or true Dice (G4); gated morphology scalars join the head input."""
    def __init__(self, arm: str, d_feat=D_FEAT, d_adapt=64, p=0.2, n_tasks=2):
        super().__init__()
        self.arm = arm
        self.adapter_g = nn.Sequential(nn.Linear(d_feat, d_adapt), nn.GELU(), nn.Dropout(p))
        self.adapter_z = nn.Sequential(nn.Linear(d_feat, d_adapt), nn.GELU(), nn.Dropout(p))
        self.relevance = nn.Parameter(torch.zeros(n_tasks, 4))
        d_in = d_adapt if arm == "G0" else 2 * d_adapt + 3
        self.heads = nn.ModuleList(nn.Linear(d_in, 1) for _ in range(n_tasks))
    def gate_weights(self, gate):
        return torch.stack([gate * torch.sigmoid(self.relevance[d])[None] for d in range(len(self.heads))], 1)   # [B,tasks,4]
    def forward(self, g, z, morph, gate):
        hg = self.adapter_g(g)
        if self.arm == "G0":
            return torch.cat([h(hg) for h in self.heads], 1)
        hz, w = self.adapter_z(z), self.gate_weights(gate); outs = []
        for d, h in enumerate(self.heads):
            wd = w[:, d]                                          # [B,4] order LV,RV,LA,RA
            anat = (wd[..., None] * hz).sum(1)
            m = morph * torch.stack([torch.minimum(wd[:, 0], wd[:, 1]), wd[:, 3], wd[:, 1]], 1)   # RV/LV ratio, RA area, RV FAC
            outs.append(h(torch.cat([hg, anat, m], 1)))
        return torch.cat(outs, 1)


class EDLHead(nn.Module):
    """U4: evidential (Dirichlet) PAH head on the G2 representation; uncertainty = K / S (vacuity), confidence = 1 − vacuity."""
    def __init__(self, base: DiagNet):
        super().__init__(); self.base = base; self.evid = nn.Linear(1, 2)
    def forward(self, g, z, morph, gate):
        logit = self.base(g, z, morph, gate)[:, :1]
        return F.softplus(self.evid(logit)) + 1.0                # alpha [B,2]


def edl_loss(alpha, y, lam):
    """Sensoy et al. Bayes-risk (squared error) + annealed KL to the flat Dirichlet."""
    S = alpha.sum(1, keepdim=True); p = alpha / S; Y = torch.stack([1 - y, y], 1)
    err = ((Y - p) ** 2 + p * (1 - p) / (S + 1)).sum(1)
    a_t = Y + (1 - Y) * alpha
    kl = (torch.lgamma(a_t.sum(1)) - torch.lgamma(torch.tensor(2.0, device=alpha.device)) - torch.lgamma(a_t).sum(1)
          + ((a_t - 1) * (torch.digamma(a_t) - torch.digamma(a_t.sum(1, keepdim=True)))).sum(1))
    return (err + lam * kl).mean()


def count_macs(model: nn.Module, x: torch.Tensor) -> int:
    """Multiply–accumulates of conv / linear / attention layers for one forward pass (hooks)."""
    macs = [0]
    def conv_hook(m, i, o):
        k = math.prod(m.kernel_size); macs[0] += o.numel() * (m.in_channels // m.groups) * k
    def lin_hook(m, i, o):
        macs[0] += o.numel() * m.in_features
    def attn_hook(m, i, o):
        q = i[0]; B, T, D = q.shape; macs[0] += B * (3 * T * D * D + 2 * T * T * D + T * D * D)
    hs = []
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Conv3d)): hs.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, nn.Linear): hs.append(m.register_forward_hook(lin_hook))
        elif isinstance(m, nn.MultiheadAttention): hs.append(m.register_forward_hook(attn_hook))
    model.eval()
    with torch.no_grad():
        model(x)
    for h in hs: h.remove()
    return macs[0]


_probe = SegModel().eval(); _x = torch.randn(1, 2, CFG.frames, 112, 112)
_rows = [("encoder", count_params(_probe.encoder)), ("temporal", count_params(_probe.temporal)), ("decoder", count_params(_probe.decoder)), ("total", count_params(_probe))]
print(f"SegModel capacity='{CFG.capacity}' (widths {CAP['widths']}, blocks {CAP['blocks']}, expand ×{CAP['expand']}, d={D_FEAT}, FPN mid {CAP['fpn_mid']}×{CAP['fpn_convs']} convs)")
print("  " + " | ".join(f"{n} {v/1e6:.3f} M" for n, v in _rows) + f" | MACs @16×112² {count_macs(_probe, _x)/1e9:.2f} G  (budget ≤ 5 M)")
assert count_params(_probe) <= 5_000_000, "parameter budget exceeded"
del _probe, _x

SegModel capacity='default' (widths (32, 64, 128, 256), blocks (2, 3, 4, 3), expand ×2, d=256, FPN mid 96×2 convs)


  encoder 1.095 M | temporal 1.596 M | decoder 0.713 M | total 3.405 M | MACs @16×112² 13.93 G  (budget ≤ 5 M)


In [5]:
# ---------------------------------------------------------------- Cell 5: Stage 0 — label-id and time-axis audit (gate); Stage 1 — audit and cache (S1, CPU)
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt


def _raw_shape(r: dict):
    if r.get("raw_shape") is not None: return tuple(r["raw_shape"])
    import nibabel as nib
    return tuple(nib.load(r["path"]).shape)


def stage0_label_and_axis_audit():
    """Evidence for the two things nobody can verify from the listing alone: which raw label id is which chamber, and which NIfTI axis is
    time. Saves a 2×5 overlay panel per raw id, prints centroid statistics and a suggested mapping from standard A4C anatomy (apex at the
    top: ventricles in the near field, RV on the image left of the LV; atria in the far field, LA on the right). Set
    CFG.cardiacnet_label_map explicitly and CFG.label_map_confirmed=True after reading it — stage 1 refuses to run otherwise."""
    require_inputs(0); out = CFG.work / "stage0"; out.mkdir(parents=True, exist_ok=True)
    recs = [r for r in discover_records() if r["source"] in ("PAH", "ASD")]
    print("[stage 0] time axis: CFG.cardiacnet_time_axis =", CFG.cardiacnet_time_axis)
    for r in recs[:5]:
        sh = _raw_shape(r); ax_t = CFG.cardiacnet_time_axis % len(sh); eq = [i for i in range(len(sh)) for j in range(i + 1, len(sh)) if sh[i] == sh[j]]
        verdict = "consistent" if (len(sh) == 3 and sh.count(sh[ax_t]) == 1 and len(eq)) else "CHECK MANUALLY"
        print(f"   {Path(r['path']).name:40s} shape {sh}  → frames on axis {ax_t} ({sh[ax_t]} frames): {verdict}")
    stats_, panels = {}, {}
    for r in recs[:80]:
        raw = r["load_raw_masks"]()
        if not raw: continue
        fr = r["load"]()
        for t, m in raw.items():
            H, W = m.shape
            for i in np.unique(m):
                if i == 0: continue
                ys, xs = np.nonzero(m == i); s = stats_.setdefault(int(i), dict(x=[], y=[], area=[]))
                s["x"].append(xs.mean() / W); s["y"].append(ys.mean() / H); s["area"].append(len(ys) / (H * W))
                if len(panels.setdefault(int(i), [])) < 10: panels[int(i)].append((fr[t], m == i, Path(r["path"]).parent.name))
    rows = pd.DataFrame([dict(raw_id=i, mean_x=np.mean(s["x"]), mean_y=np.mean(s["y"]), mean_area=np.mean(s["area"]), n_frames=len(s["x"])) for i, s in sorted(stats_.items())])
    print("\n[stage 0] raw label-id statistics (x,y normalised to image width/height; area = fraction of image):\n", rows.round(3).to_string(index=False))
    if len(rows) >= 4:
        by_y = rows.sort_values("mean_y"); vent, atr = by_y.iloc[:2], by_y.iloc[2:4]
        sug = {int(vent.sort_values("mean_x").raw_id.iloc[1]): 1, int(vent.sort_values("mean_x").raw_id.iloc[0]): 2, int(atr.sort_values("mean_x").raw_id.iloc[1]): 3, int(atr.sort_values("mean_x").raw_id.iloc[0]): 4}
        print(f"[stage 0] suggested CFG.cardiacnet_label_map (raw id → 1 LV, 2 RV, 3 LA, 4 RA), assuming apex-up A4C with RV on the image left: {dict(sorted(sug.items()))}")
        print(f"          current CFG.cardiacnet_label_map: {CFG.cardiacnet_label_map} → {'MATCHES' if dict(sorted(sug.items())) == dict(sorted(CFG.cardiacnet_label_map.items())) else 'DIFFERS — inspect the panels'}")
    for i, items in panels.items():
        fig, axs = plt.subplots(2, 5, figsize=(15, 6))
        for ax, (f, m, name) in zip(axs.ravel(), items):
            ax.imshow(f, cmap="gray"); ax.imshow(np.ma.masked_where(~m, m), cmap="autumn", alpha=0.5); ax.set_title(f"{name} id={i}", fontsize=8); ax.axis("off")
        for ax in axs.ravel()[len(items):]: ax.axis("off")
        plt.tight_layout(); plt.savefig(out / f"label_id_{i}.png", dpi=80); plt.close()
    rows.to_csv(out / "label_id_stats.csv", index=False); print(f"[stage 0] panels → {out} ; then set CFG.cardiacnet_label_map and K1_LABEL_MAP_CONFIRMED=1")


if 0 in CFG.stages:
    stage0_label_and_axis_audit()


def compute_flow_magnitude(frames: np.ndarray) -> Optional[np.ndarray]:
    """TV-L1 flow if opencv-contrib is present, Farneback otherwise; None if cv2 is missing. Returns |flow| per frame (fp16)."""
    try:
        import cv2
    except ImportError:
        return None
    tvl1 = cv2.optflow.DualTVL1OpticalFlow_create() if hasattr(cv2, "optflow") else None
    out = np.zeros(frames.shape, np.float16)
    for t in range(1, len(frames)):
        a, b = frames[t - 1], frames[t]
        fl = tvl1.calc(a, b, None) if tvl1 is not None else cv2.calcOpticalFlowFarneback(a, b, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        out[t] = np.linalg.norm(fl, axis=-1).astype(np.float16)
    return out


def stage1_audit_and_cache():
    require_inputs(1)
    if not SMOKE_TEST and not CFG.label_map_confirmed:
        raise RuntimeError("Stage 1 refused: run Stage 0, set CFG.cardiacnet_label_map explicitly and set K1_LABEL_MAP_CONFIRMED=1")
    out = stage_dir(CFG.ds_cache, create=True); (out / "clips").mkdir(exist_ok=True); (out / "flow").mkdir(exist_ok=True)
    recs = discover_records()
    # 1) SHA-256 dedup and contradictory-label quarantine ----------------------------------------------------------
    by_hash: Dict[str, List[dict]] = {}
    for r in recs: by_hash.setdefault(r["sha256"], []).append(r)
    audit, keep = [], []
    for h, grp in by_hash.items():
        labels = {g["label"] for g in grp if g["label"] is not None}
        status = "quarantined_contradictory" if len(labels) > 1 else "kept"
        for k, g in enumerate(grp):
            s = status if status != "kept" else ("kept" if k == 0 else "dropped_duplicate")
            audit.append(dict(id=g["id"], path=g["path"], source=g["source"], label=g["label"], group=g["group"], sha256=h, status=s))
            if s == "kept": keep.append(g)
    audit = pd.DataFrame(audit); audit.to_csv(out / "audit.csv", index=False)
    print("[stage 1] audit:", audit.status.value_counts().to_dict())
    # 2) acquisition groups: evidence, gate, then ONE stratified group fold draw over the union of both branches --------
    man = pd.DataFrame([dict(id=r["id"], source=r["source"], branch=r["branch"], class_folder=r["class_folder"], token=r["token"], path=r["path"], group=r["group"], label=r["label"], sha256=r["sha256"]) for r in keep])
    man["group_regex"] = CFG.group_regex; man["group_scope"] = CFG.group_scope
    lab = man[man.source.isin(["PAH", "ASD"])]; sizes = lab.groupby("group").size(); n_groups = int(lab.group.nunique())
    print(f"[stage 1] grouping scope='{CFG.group_scope}', regex={CFG.group_regex!r}: {len(lab)} labelled videos in {n_groups} groups")
    print("   20 most frequent tokens:", lab.token.value_counts().head(20).to_dict())
    print("   group-size histogram (size: n_groups):", sizes.value_counts().sort_index().to_dict())
    if not SMOKE_TEST:
        assert 250 <= n_groups <= len(lab), f"n_groups={n_groups} outside [250, n_videos={len(lab)}]: fix CFG.group_regex / CFG.group_scope before caching"
        assert CFG.grouping_confirmed, "Stage 1 refused: inspect the token/group evidence above and set K1_GROUPING_CONFIRMED=1"
    man["fold"] = -1
    strat = (lab.branch + "_" + lab.label.astype(int).astype(str))
    skf = StratifiedGroupKFold(n_splits=CFG.n_folds, shuffle=True, random_state=0)
    for f, (_, te) in enumerate(skf.split(lab, strat, lab.group)):
        man.loc[lab.index[te], "fold"] = f
    assert man[man.fold >= 0].groupby("group").fold.nunique().max() == 1, "a group straddles folds"
    man["y_pah"] = np.where(man.source == "PAH", man.label, np.nan); man["y_asd"] = np.where(man.source == "ASD", man.label, np.nan)
    man["has_masks"] = False
    # 3) cache: K samplings per video (0 = annotation-preserving at student + teacher size; 1..K-1 = phase offsets), flow on sampling 0 --
    rows, flow_ok, K = [], True, CFG.n_samplings
    for r in keep:
        fr = r["load"](); masks = r["masks"] if "masks" in r else r["load_masks"]()
        T = fr.shape[0]; idx = [annotation_preserving_indices(T, CFG.frames, list(masks.keys()))] + [phase_offset_indices(T, CFG.frames, k, K) for k in range(1, K)]
        ssl = np.stack([resize_stack(fr[i], CFG.size).round().clip(0, 255).astype(np.uint8) for i in idx])
        tea = np.stack([resize_stack(fr[i], CFG.teacher_size).round().clip(0, 255).astype(np.uint8) for i in idx])
        m = np.full((CFG.frames, CFG.size, CFG.size), IGNORE, np.uint8)
        for j, t in enumerate(idx[0]):
            if int(t) in masks: m[j] = resize_stack(masks[int(t)][None], CFG.size, nearest=True)[0].astype(np.uint8)
        np.savez_compressed(out / "clips" / f"{r['id']}.npz", frames=ssl[0], frames_ssl=ssl, frames_teacher=tea, masks=m, src_idx=np.stack(idx))
        man.loc[man.id == r["id"], "has_masks"] = bool(masks)
        rows.append(dict(id=r["id"], n_src_frames=T, annotated_src=json.dumps(sorted(int(t) for t in masks)), annotated_cache=json.dumps([int(j) for j, t in enumerate(idx[0]) if int(t) in masks])))
        if flow_ok:
            fl = compute_flow_magnitude(ssl[0])
            if fl is None: flow_ok = False
            else: np.save(out / "flow" / f"{r['id']}.npy", fl)
    pd.DataFrame(rows).to_csv(out / "annotated_frames.csv", index=False)
    if not flow_ok: print("[stage 1] cv2 missing: flow fields skipped (only the optional flow-channel arm needs them)")
    # 4) per-fold intensity normalisation from training videos only ---------------------------------------------
    stats_ = {}
    for f in list(range(CFG.n_folds)) + [-1]:
        ids = man[(man.fold != f) & (man.source.isin(["PAH", "ASD"]))].id if f >= 0 else man.id
        vals, dvals, fvals = [], [], []
        for i in ids:
            g = np.load(out / "clips" / f"{i}.npz")["frames"].astype(np.float32) / 255.0
            vals.append(g.ravel()[::7]); dvals.append(np.abs(g[2:] - g[:-2]).ravel()[::7])
            if flow_ok: fvals.append(np.load(out / "flow" / f"{i}.npy").astype(np.float32).ravel()[::7])
        v, d = np.concatenate(vals), np.concatenate(dvals)
        stats_[str(f)] = dict(mean=float(v.mean()), std=float(v.std() + 1e-6), diff_std=float(d.std() + 1e-6), flow_std=(float(np.concatenate(fvals).std() + 1e-6) if fvals else 1.0))
    (out / "norm_stats.json").write_text(json.dumps(stats_, indent=1))
    man.to_csv(out / "manifest.csv", index=False)
    (out / "manifest.sha256").write_text(hashlib.sha256((out / "manifest.csv").read_bytes()).hexdigest())
    write_version(CFG.ds_cache, dict(n_videos=int(len(man)), n_labelled=int(len(lab)), n_groups=n_groups, group_regex=CFG.group_regex, group_scope=CFG.group_scope, n_samplings=K,
                                     sizes=dict(student=CFG.size, teacher=CFG.teacher_size), label_map=CFG.cardiacnet_label_map, manifest_sha256=(out / "manifest.sha256").read_text()))
    print(f"[stage 1] cached {len(man)} videos × {K} samplings ({(man.source=='PAH').sum()} PAH-set: {int(man.y_pah.sum())} PAH / {int((man.y_pah==0).sum())} non-PAH; "
          f"{(man.source=='ASD').sum()} ASD-set; {(~man.source.isin(['PAH','ASD'])).sum()} extras) in {n_groups} acquisition groups → {out}")
    print(man.groupby(["source", "fold"]).size().unstack(fill_value=0))


if 1 in CFG.stages:
    stage1_audit_and_cache()


def load_cache():
    """Manifest, normalisation stats and paths for the stages that follow."""
    d = stage_dir(CFG.ds_cache); man = pd.read_csv(d / "manifest.csv")
    assert (d / "manifest.sha256").read_text() == hashlib.sha256((d / "manifest.csv").read_bytes()).hexdigest(), "manifest hash mismatch"
    return d, man, json.loads((d / "norm_stats.json").read_text())

[stage 0] depends on: nothing


[stage 0] time axis: CFG.cardiacnet_time_axis = -1
   acq001_0.nii.gz                          shape (20, 96, 96)  → frames on axis 2 (96 frames): CHECK MANUALLY
   acq002_0.nii.gz                          shape (20, 96, 96)  → frames on axis 2 (96 frames): CHECK MANUALLY
   acq002_1.nii.gz                          shape (20, 96, 96)  → frames on axis 2 (96 frames): CHECK MANUALLY
   acq003_0.nii.gz                          shape (20, 96, 96)  → frames on axis 2 (96 frames): CHECK MANUALLY
   acq003_1.nii.gz                          shape (20, 96, 96)  → frames on axis 2 (96 frames): CHECK MANUALLY

[stage 0] raw label-id statistics (x,y normalised to image width/height; area = fraction of image):
  raw_id  mean_x  mean_y  mean_area  n_frames
      1    0.68   0.629      0.075       330
      2    0.34   0.614      0.079       330
      3    0.66   0.299      0.033       330
      4    0.35   0.299      0.029       330
[stage 0] suggested CFG.cardiacnet_label_map (raw id → 1 LV, 2 RV, 

[stage 0] panels → k1_work/stage0 ; then set CFG.cardiacnet_label_map and K1_LABEL_MAP_CONFIRMED=1
[stage 1] depends on: nothing


[stage 1] audit: {'kept': 71, 'quarantined_contradictory': 2, 'dropped_duplicate': 1}
[stage 1] grouping scope='folder', regex='^(.*?)(?:[_\\-][A-Za-z]?\\d+)?$': 63 labelled videos in 44 groups
   20 most frequent tokens: {'acq008': 3, 'acq010': 3, 'acq011': 3, 'acq013': 3, 'acq016': 3, 'acq017': 3, 'acq019': 3, 'acq024': 3, 'acq027': 3, 'acq030': 3, 'acq002': 2, 'acq003': 2, 'acq004': 2, 'acq005': 2, 'acq006': 2, 'acq007': 2, 'acq012': 2, 'acq018': 2, 'acq020': 2, 'acq021': 2}
   group-size histogram (size: n_groups): {1: 27, 2: 15, 3: 2}


[stage 1] cached 71 videos × 4 samplings (47 PAH-set: 29 PAH / 18 non-PAH; 16 ASD-set; 8 extras) in 44 acquisition groups → k1_work/cardiacnet-cache-v1
fold    -1   0   1   2   3   4
source                        
ASD      0   4   3   4   2   3
CAMUS    4   0   0   0   0   0
HMCQU    4   0   0   0   0   0
PAH      0   9  10   9  10   9


In [6]:
# ---------------------------------------------------------------- Cell 6: Stage 2 — frozen teacher latents (S2)
class SyntheticTeacher(nn.Module):
    """Frozen random 3-D CNN used only in validation mode so that the distillation code path runs without EchoJEPA weights."""
    input_res, dim, licence = 64, 256, "n/a (synthetic)"
    def __init__(self):
        super().__init__(); torch.manual_seed(1234)
        self.net = nn.Sequential(nn.Conv3d(3, 32, (3, 5, 5), stride=(1, 2, 2), padding=(1, 2, 2)), nn.GELU(), nn.Conv3d(32, 64, 3, stride=(1, 2, 2), padding=1), nn.GELU(),
                                 nn.Conv3d(64, 128, 3, stride=(2, 2, 2), padding=1), nn.GELU(), nn.AdaptiveAvgPool3d(1), nn.Flatten(), nn.Linear(128, self.dim))
        for p in self.parameters(): p.requires_grad_(False)
    def forward(self, x):                                       # x [B,3,T,H,W] in [0,1]
        return self.net(x - 0.5)


class AdapterTeacher(nn.Module):
    """Real teacher through an adapter module from teacher_adapters/ (echojepa_l.py — MIT; panecho.py — CC BY-NC-SA), copied into the
    weights dataset. Each adapter exposes build(weights_dir) -> (module, input_res, dim, preprocess, licence); module(x) returns the pooled
    clip embedding for x = preprocess(clip in [0,1], [B,3,16,input_res,input_res]); module.tokens(x) returns the token grid when available."""
    def __init__(self, name: str, weights_dir: Path):
        super().__init__()
        cands = [Path(weights_dir) / f"{name}.py", Path(weights_dir) / "teacher_adapters" / f"{name}.py", Path("teacher_adapters") / f"{name}.py", Path("../teacher_adapters") / f"{name}.py"]
        adapter = next((c for c in cands if c.exists()), None)
        if adapter is None: raise FileNotFoundError(f"teacher adapter {name}.py not found in {[str(c) for c in cands]}")
        import importlib.util
        spec = importlib.util.spec_from_file_location(f"teacher_adapter_{name}", adapter); mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
        self.model, self.input_res, self.dim, self.preprocess, self.licence = mod.build(Path(weights_dir)); self.model.eval()
        for p in self.model.parameters(): p.requires_grad_(False)
    def forward(self, x): return self.model(self.preprocess(x))


def load_teacher() -> nn.Module:
    t = SyntheticTeacher() if SMOKE_TEST else AdapterTeacher(CFG.teacher_name, CFG.teacher_weights)
    return t.to(DEVICE).eval()


@torch.no_grad()
def stage2_teacher_latents():
    """Every 16-frame sampling of every clip (CardiacNet + CAMUS + HMC-QU) at the cached teacher resolution; no sub-cropping (A7)."""
    require_inputs(2)
    cache, man, _ = load_cache(); out = stage_dir(CFG.ds_latents, create=True)
    teacher = load_teacher(); K = CFG.n_samplings; ids, ks, lat = [], [], []
    for b0 in range(0, len(man), CFG.batch):
        chunk = man.iloc[b0:b0 + CFG.batch]
        g = torch.stack([torch.from_numpy(np.load(cache / "clips" / f"{i}.npz")["frames_teacher"]) for i in chunk.id]).float().div_(255.0).to(DEVICE)   # [B,K,T,S,S]
        for k in range(K):
            clip = g[:, k]                                       # [B,T,S,S]
            if clip.shape[-1] != teacher.input_res:              # only when the cache was built at another teacher size
                clip = F.interpolate(clip.reshape(-1, 1, *clip.shape[-2:]), size=(teacher.input_res, teacher.input_res), mode="bilinear", align_corners=False).view(len(chunk), CFG.frames, teacher.input_res, teacher.input_res)
            x = clip[:, None].repeat(1, 3, 1, 1, 1)              # [B,3,T,S,S]
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                z = teacher(x).float()
            ids += list(chunk.id); ks += [k] * len(chunk); lat.append(z.cpu().half().numpy())
    lat = np.concatenate(lat)
    np.savez_compressed(out / "latents.npz", id=np.array(ids), sampling=np.array(ks), latent=lat)
    write_version(CFG.ds_latents, dict(teacher=type(teacher).__name__, teacher_name=CFG.teacher_name, licence=getattr(teacher, "licence", "see adapter"), dim=int(lat.shape[1]), n_clips=int(len(man)), samplings=K, input_res=teacher.input_res, frames=CFG.frames))
    print(f"[stage 2] {len(ids)} latents ({len(man)} clips × {K} samplings × {CFG.frames} frames @ {teacher.input_res} px, dim {lat.shape[1]}) from {type(teacher).__name__} → {out}")


if 2 in CFG.stages:
    stage2_teacher_latents()

[stage 2] depends on: ['cardiacnet-cache-v1']
   cardiacnet-cache-v1    -> k1_work/cardiacnet-cache-v1  (2026-09-29T13:37:36)


[stage 2] 284 latents (71 clips × 4 samplings × 16 frames @ 64 px, dim 256) from SyntheticTeacher → k1_work/teacher-latents-v1


In [7]:
# ---------------------------------------------------------------- Cell 7: Stage 3 — student encoder distillation (S3) → D1 init
class DistillDataset(torch.utils.data.Dataset):
    """Two of the K 16-frame samplings of the same clip per item (positives, EchoCLR-style; same-group clips are positives too) with their
    teacher latents. Student input at student resolution."""
    def __init__(self, man, cache, norm, latents):
        self.man = man.reset_index(drop=True); self.cache = cache; self.norm = norm; self.K = CFG.n_samplings
        self.key = {(i, int(k)): j for j, (i, k) in enumerate(zip(latents["id"], latents["sampling"]))}; self.lat = latents["latent"].astype(np.float32)
        self.gid = {g: k for k, g in enumerate(sorted(self.man.group.unique()))}
    def __len__(self): return len(self.man)
    def _clip(self, c):
        gi = (c - self.norm["mean"]) / self.norm["std"]; return torch.stack([gi, ClipDataset.frame_difference(c) / self.norm["diff_std"]])
    def __getitem__(self, i):
        r = self.man.iloc[i]; g = torch.from_numpy(np.load(self.cache / "clips" / f"{r.id}.npz")["frames_ssl"]).float() / 255.0     # [K,T,H,W]
        ka, kb = random.sample(range(self.K), 2)
        if CFG.flip_mode == "swap" and random.random() < 0.5: g = g.flip(-1)
        return (self._clip(g[ka]), torch.from_numpy(self.lat[self.key[(r.id, ka)]]), self._clip(g[kb]), torch.from_numpy(self.lat[self.key[(r.id, kb)]]), self.gid[r.group])


def tube_mask(x: torch.Tensor, ratio: float, patch=8) -> torch.Tensor:
    """Masked-latent prediction: zero a random `ratio` of (T × patch × patch) tubes of the student input."""
    B, C, T, H, W = x.shape; gh, gw = H // patch, W // patch
    keep = (torch.rand(B, 1, 1, gh, gw, device=x.device) >= ratio).float()
    return x * F.interpolate(keep.flatten(0, 1), size=(H, W), mode="nearest").view(B, 1, 1, H, W)


def supcon_loss(z: torch.Tensor, labels: torch.Tensor, tau=0.1) -> torch.Tensor:
    """Supervised contrastive loss with same-study (acquisition-group) positives; items without a positive are skipped."""
    z = F.normalize(z, dim=1); sim = z @ z.T / tau
    eye = torch.eye(len(z), device=z.device, dtype=torch.bool)
    pos = (labels[:, None] == labels[None]) & ~eye
    logp = sim.masked_fill(eye, -1e4).log_softmax(1)
    n_pos = pos.sum(1); valid = n_pos > 0
    return -(logp * pos).sum(1)[valid].div(n_pos[valid]).mean() if valid.any() else z.sum() * 0.0


class DistillNet(nn.Module):
    def __init__(self, core: SegModel, dim: int):
        super().__init__(); self.core = core; self.proj = nn.Sequential(nn.Linear(D_FEAT, 256), nn.GELU(), nn.Linear(256, dim))
    def forward(self, x):
        return self.proj(self.core.forward_features(x)[1].mean(1))


def stage3_distill():
    require_inputs(3)
    cache, man, norm_all = load_cache(); norm = norm_all["-1"]; out = stage_dir(CFG.ds_student, create=True)
    lat = np.load(stage_dir(CFG.ds_latents) / "latents.npz"); D = lat["latent"].shape[1]
    ds = DistillDataset(man, cache, norm, lat)
    dl = torch.utils.data.DataLoader(ds, batch_size=max(2, CFG.batch // 2), shuffle=True, drop_last=True, num_workers=CFG.num_workers)
    seed_everything(0); core = SegModel(); model = parallelise(DistillNet(core, D))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=max(1, CFG.distill_epochs * len(dl)), pct_start=0.1)
    scaler = make_scaler(); ck = out / "ckpt_distill.pt"; log = RunLog(out); start = 0
    if ck.exists():
        start, _ = load_ckpt(ck, model, opt, sched); print(f"[stage 3] resuming from epoch {start}")
    reset_peak_mem()
    for ep in range(start, CFG.distill_epochs):
        model.train(); tot, n = 0.0, 0
        with Timer() as tm:
            for xa, la, xb, lb, gid in dl:
                x = torch.cat([xa, xb]).to(DEVICE); tgt = torch.cat([la, lb]).to(DEVICE); gid = torch.cat([gid, gid]).to(DEVICE)
                with autocast_ctx():
                    z = model(tube_mask(x, CFG.mask_ratio)).float()
                    loss_lat = F.l1_loss(z, tgt) + (1 - F.cosine_similarity(z, tgt, dim=1)).mean()
                    loss = loss_lat + 0.5 * supcon_loss(z, gid)
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 3.0); scaler.step(opt); scaler.update(); sched.step()
                tot += loss.item() * len(x); n += len(x)
        save_ckpt(ck, model, opt, sched, ep + 1)
        log.append(f"distill_ep{ep+1:03d}", dict(stage=3, epoch=ep + 1, loss=tot / max(n, 1), sec=tm.dt, peak_mem_mb=peak_mem_mb()))
        print(f"[stage 3] epoch {ep+1}/{CFG.distill_epochs} loss {tot/max(n,1):.4f} ({tm.dt:.0f}s)")
    torch.save({"encoder": core.encoder.state_dict(), "temporal": core.temporal.state_dict()}, out / "student_encoder.pt")
    write_version(CFG.ds_student, dict(params_core=count_params(core.encoder) + count_params(core.temporal), teacher_dim=D, capacity=CFG.capacity))
    print(f"[stage 3] student init saved → {out/'student_encoder.pt'}")


def init_core(init: str) -> SegModel:
    """D1: encoder + temporal module from the distilled student. D0: no distillation (Kaiming init, or a user-supplied
    state dict at $D0_WEIGHTS — an ImageNet/Kinetics stem cannot be mapped onto the depthwise-separable blocks automatically)."""
    core = SegModel()
    if init == "D1":
        sd = torch.load(stage_dir(CFG.ds_student) / "student_encoder.pt", map_location="cpu", weights_only=False)
        core.encoder.load_state_dict(sd["encoder"]); core.temporal.load_state_dict(sd["temporal"])
    elif init == "D0" and os.environ.get("D0_WEIGHTS"):
        missing = core.load_state_dict(torch.load(os.environ["D0_WEIGHTS"], map_location="cpu", weights_only=False), strict=False)
        print("[init D0] loaded external stem; missing keys:", len(missing.missing_keys))
    return core


if 3 in CFG.stages:
    stage3_distill()

[stage 3] depends on: ['cardiacnet-cache-v1', 'teacher-latents-v1']
   cardiacnet-cache-v1    -> k1_work/cardiacnet-cache-v1  (2026-09-29T13:37:36)
   teacher-latents-v1     -> k1_work/teacher-latents-v1  (2026-09-29T13:37:37)


[stage 3] epoch 1/2 loss 0.5722 (26s)


[stage 3] epoch 2/2 loss 0.3536 (22s)
[stage 3] student init saved → k1_work/student-init-v1/student_encoder.pt


In [8]:
# ---------------------------------------------------------------- Cell 8: Stage 4 — segmentation, cross-fitted quality targets, cached features (S4–S5)
# Design (A1/A2): the three inner cross-fit segmenters exist ONLY to produce out-of-sample masks on which the quality estimator is fitted.
# The cached diagnostic features (g, z, morph, ent, q̂) of EVERY video of fold k — training videos with or without masks, and test
# videos — come from the single FINAL frozen core of that fold, so adapters, gate and heads train and test in one latent space.
# Training-video q̂ is therefore mildly optimistic (the final core saw those videos' masks); this is the accepted cost of one frozen
# deployable core and is recorded in summary.json.

def train_segmenter(train_df: pd.DataFrame, cache: Path, norm: dict, init: str, ckpt: Path, epochs: int, log: RunLog, run_id: str, seed=0) -> SegModel:
    """Trains encoder + temporal + FPN on annotated frames; checkpoints every epoch and resumes from the latest one."""
    seed_everything(seed); model = parallelise(init_core(init))
    ds = ClipDataset(train_df, cache, norm, augment=True)
    dl = torch.utils.data.DataLoader(ds, batch_size=CFG.batch, shuffle=True, drop_last=len(ds) > CFG.batch, num_workers=CFG.num_workers)
    core = unwrap(model)
    opt = torch.optim.AdamW([{"params": core.core_parameters(), "lr": CFG.lr_core if init == "D1" else 1e-3}, {"params": core.decoder.parameters(), "lr": 1e-3}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs * len(dl)))
    scaler = make_scaler(); start = 0
    if ckpt.exists():
        start, _ = load_ckpt(ckpt, model, opt, sched)
    reset_peak_mem(); t0 = time.time(); last = float("nan")
    for ep in range(start, epochs):
        model.train(); tot, n = 0.0, 0
        for x, m, _, _ in dl:
            x, m = x.to(DEVICE), m.to(DEVICE)
            with autocast_ctx():
                logits, _, _ = model(x); loss = seg_loss(logits, m)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 3.0); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(x); n += len(x)
        last = tot / max(n, 1); save_ckpt(ckpt, model, opt, sched, ep + 1)
    log.append(run_id, dict(stage=4, init=init, epochs=epochs, n_train=len(train_df), final_loss=last, sec=time.time() - t0, sec_per_epoch=(time.time() - t0) / max(1, epochs - start), peak_mem_mb=peak_mem_mb(), params=count_params(core)))
    return core.eval()


# ---------------- per-chamber quality features ---------------------------------------------------------------------
def radial_profile(mask: np.ndarray, K=36, centre=None):
    ys, xs = np.nonzero(mask)
    if len(ys) < 3:
        return None, centre
    cy, cx = (ys.mean(), xs.mean()) if centre is None else centre
    ang = np.arctan2(ys - cy, xs - cx); rad = np.hypot(ys - cy, xs - cx)
    bins = ((ang + np.pi) / (2 * np.pi) * K).astype(int) % K
    r = np.zeros(K); np.maximum.at(r, bins, rad)
    return r, (cy, cx)


def chamber_frame_features(prob: np.ndarray, pred: np.ndarray, prev_pred: Optional[np.ndarray], next_pred: Optional[np.ndarray], c: int) -> dict:
    """Mask area, solidity, boundary-curvature statistics, softmax entropy/confidence, temporal mask consistency,
    CASUS-style contour-sample variance across probability thresholds. prob is [C,H,W]."""
    m = pred == c; area = float(m.mean()); f = dict(area=area)
    ys, xs = np.nonzero(m)
    if len(ys) >= 4:
        try:
            hull = ConvexHull(np.stack([ys, xs], 1)); f["solidity"] = float(len(ys) / max(hull.volume, 1.0))
        except Exception:
            f["solidity"] = 0.0
    else:
        f["solidity"] = 0.0
    r, cen = radial_profile(m)
    if r is not None:
        rn = r / (r.mean() + 1e-6); k = np.roll(rn, 1) - 2 * rn + np.roll(rn, -1)
        f.update(curv_mean=float(np.abs(k).mean()), curv_std=float(k.std()), curv_max=float(np.abs(k).max()))
        profiles = [radial_profile(prob[c] >= thr, centre=cen)[0] for thr in (0.3, 0.5, 0.7)]
        profiles = [p for p in profiles if p is not None]
        f["casus_var"] = float((np.stack(profiles).var(0) / (r.mean() ** 2 + 1e-6)).mean()) if len(profiles) > 1 else 1.0
    else:
        f.update(curv_mean=0.0, curv_std=0.0, curv_max=0.0, casus_var=1.0)
    ent = -(prob * np.log(prob + 1e-8)).sum(0) / math.log(prob.shape[0])
    f["entropy"] = float(ent[m].mean() if m.any() else ent.mean()); f["confidence"] = float(prob[c][m].mean() if m.any() else prob[c].mean())
    tc = [dice_binary(m, o == c) for o in (prev_pred, next_pred) if o is not None]
    f["temporal_dice"] = float(np.mean(tc)) if tc else 1.0
    return f


QUALITY_FEATS = ["area", "solidity", "curv_mean", "curv_std", "curv_max", "casus_var", "entropy", "confidence", "temporal_dice"]


class QualityEstimator:
    """Per-chamber gradient-boosted regressors (CPU) on morphological/uncertainty features + PCA of detached pooled image features.
    Target: per-chamber Dice of the predicted mask against the manual mask. Fitted on cross-fitted (out-of-sample) predictions only."""
    name = "hgb_morphology"
    def __init__(self, n_pca=8):
        from sklearn.decomposition import PCA
        from sklearn.ensemble import HistGradientBoostingRegressor
        self.pca = PCA(n_components=n_pca, random_state=0); self.n_pca = n_pca
        self.models = {c: HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0, random_state=0) for c in range(1, N_CLS)}
    def _X(self, rows: pd.DataFrame, feats: np.ndarray, fit=False):
        pcs = self.pca.fit_transform(feats) if fit else self.pca.transform(feats)
        return np.concatenate([rows[QUALITY_FEATS].to_numpy(np.float32), pcs.astype(np.float32)], 1)
    def fit(self, rows: pd.DataFrame, feats: np.ndarray, mlp_feats=None):
        X = self._X(rows, feats, fit=True)
        for c, m in self.models.items():
            sel = (rows.chamber == c).to_numpy()
            if sel.sum() >= 10: m.fit(X[sel], rows.dice.to_numpy()[sel])
            else: m.fit(X[:2], np.array([0.5, 0.5]))
        return self
    def predict(self, rows: pd.DataFrame, feats: np.ndarray, mlp_feats=None) -> np.ndarray:
        X = self._X(rows, feats); q = np.zeros(len(rows), np.float32)
        for c, m in self.models.items():
            sel = (rows.chamber == c).to_numpy()
            if sel.any(): q[sel] = m.predict(X[sel])
        return np.clip(q, 0, 1)


class QualityHeadMLP:
    """GPU alternative (B4): an MLP on the per-chamber mask-weighted pooled bottleneck feature + entropy + area + temporal Dice, trained on
    the same out-of-sample targets. Needs no CPU morphology, so it is the deployable choice when the CPU feature extraction is slow."""
    name = "mlp_head"
    def __init__(self, d_in, epochs=300):
        self.net = nn.Sequential(nn.Linear(d_in + 3 + 4, 128), nn.GELU(), nn.Linear(128, 64), nn.GELU(), nn.Linear(64, 1)); self.epochs = epochs; self.mu = None
    def _X(self, rows, mlp_feats):
        X = np.concatenate([mlp_feats, rows[["entropy", "area", "temporal_dice"]].to_numpy(np.float32), np.eye(4, dtype=np.float32)[rows.chamber.to_numpy() - 1]], 1)
        if self.mu is None: self.mu, self.sd = X.mean(0), X.std(0) + 1e-6
        return torch.tensor((X - self.mu) / self.sd, dtype=torch.float32)
    def fit(self, rows, feats, mlp_feats):
        torch.manual_seed(0); X = self._X(rows, mlp_feats); y = torch.tensor(rows.dice.to_numpy(np.float32))[:, None]
        opt = torch.optim.Adam(self.net.parameters(), lr=2e-3, weight_decay=1e-4)
        for _ in range(self.epochs):
            opt.zero_grad(); loss = F.mse_loss(self.net(X), y); loss.backward(); opt.step()
        return self
    def predict(self, rows, feats, mlp_feats):
        with torch.no_grad(): return np.clip(self.net(self._X(rows, mlp_feats)).squeeze(1).numpy(), 0, 1)


@torch.no_grad()
def infer_videos(model: SegModel, df: pd.DataFrame, cache: Path, norm: dict, out_dir: Optional[Path], tag: str):
    """Frozen-core inference. Per video: masks + pooled features (g, per-chamber z, morphology, entropy, true Dice where masks exist).
    Returns per-(frame, chamber) quality rows, the pooled frame features (for the HGB estimator's PCA), the per-row pooled chamber
    features (for the MLP head), per-video rows, and the CPU seconds spent on quality features per clip."""
    model = model.to(DEVICE).eval(); ds = ClipDataset(df, cache, norm)
    dl = torch.utils.data.DataLoader(ds, batch_size=CFG.batch, shuffle=False, num_workers=CFG.num_workers)
    if out_dir is not None: (out_dir / "preds").mkdir(parents=True, exist_ok=True)
    rows, frame_feats, mlp_feats, video_rows, cpu_sec = [], [], [], [], 0.0
    for x, m, y, idx in dl:
        with autocast_ctx():
            logits, f4, _ = model(x.to(DEVICE))
        prob = logits[-1].float().softmax(1).cpu().numpy(); f4 = f4.float()
        pred = prob.argmax(1).astype(np.uint8); m = m.numpy()
        p_small = F.adaptive_avg_pool3d(torch.from_numpy(prob).to(f4.device), f4.shape[-3:])                          # [B,C,T,h,w]
        z = torch.einsum("bcthw,bdthw->bcd", p_small, f4) / (p_small.sum(dim=(2, 3, 4))[..., None] + 1e-6)             # clip-level mask-weighted pooling
        zt = torch.einsum("bcthw,bdthw->btcd", p_small, f4) / (p_small.sum(dim=(3, 4))[..., None].transpose(1, 2) + 1e-6)   # per-frame per-chamber [B,T,C,D]
        g = f4.mean(dim=(2, 3, 4)).cpu().numpy(); z = z[:, 1:].cpu().numpy(); zt = zt[:, :, 1:].cpu().numpy(); ft = f4.mean(dim=(3, 4)).transpose(1, 2).cpu().numpy()
        for b in range(len(idx)):
            r = df.iloc[int(idx[b])]; T = pred.shape[1]
            areas = np.stack([(pred[b] == c).mean(axis=(1, 2)) for c in range(N_CLS)], 1)          # [T,C]
            ed = int(areas[:, 1].argmax()); rv = areas[:, 2]
            morph = np.array([min(areas[ed, 2] / (areas[ed, 1] + 1e-4), 5.0), areas[ed, 4], (rv.max() - rv.min()) / (rv.max() + 1e-6)], np.float32)
            cx = [np.nonzero(pred[b] == c)[2].mean() if (pred[b] == c).any() else np.nan for c in (1, 2)]              # LV, RV centroid x (orientation audit)
            ent_v = np.zeros(4); dice_v = np.full(4, np.nan); dice_acc = {c: [] for c in range(1, N_CLS)}
            t0 = time.perf_counter()
            for t in range(T):
                annotated = bool((m[b, t] != IGNORE).any())
                for c in range(1, N_CLS):
                    f = chamber_frame_features(prob[b, :, t], pred[b, t], pred[b, t - 1] if t > 0 else None, pred[b, t + 1] if t + 1 < T else None, c)
                    d = dice_binary(pred[b, t] == c, m[b, t] == c) if annotated else np.nan
                    msd, hd95 = surface_distances(pred[b, t] == c, m[b, t] == c) if annotated else (np.nan, np.nan)
                    rows.append(dict(id=r.id, t=t, chamber=c, annotated=annotated, dice=d, msd=msd, hd95=hd95, **f)); frame_feats.append(ft[b, t]); mlp_feats.append(zt[b, t, c - 1])
                    ent_v[c - 1] += f["entropy"] / T
                    if annotated: dice_acc[c].append(d)
            cpu_sec += time.perf_counter() - t0
            for c in range(1, N_CLS):
                if dice_acc[c]: dice_v[c - 1] = float(np.mean(dice_acc[c]))
            if out_dir is not None:
                np.savez_compressed(out_dir / "preds" / f"{r.id}.npz", mask=pred[b], g=g[b], z=z[b], morph=morph, ent=ent_v, dice=dice_v, source=tag)
            video_rows.append(dict(id=r.id, group=r.group, fold=r.fold, y_pah=r.y_pah, y_asd=r.y_asd, has_masks=bool(r.has_masks), source_pred=tag, lv_cx=cx[0], rv_cx=cx[1]))
    return pd.DataFrame(rows), np.stack(frame_feats), np.stack(mlp_feats), pd.DataFrame(video_rows), cpu_sec / max(1, len(df))


def _spear(a, b):
    return stats.spearmanr(a, b)[0] if (len(a) > 2 and np.std(a) > 0 and np.std(b) > 0) else np.nan


def stage4_segmentation():
    require_inputs(4)
    cache, man, norms = load_cache(); cores = stage_dir(CFG.ds_cores, create=True); feats_out = stage_dir(CFG.ds_feats, create=True)
    log = RunLog(cores); manifest = RunManifest(cores / "runs_manifest.json")
    lab = man[man.source.isin(["PAH", "ASD"])].reset_index(drop=True)
    for init in CFG.inits:
        do_inner = (init == "D1") or CFG.d0_full            # B1: D0 runs the final segmenter only (H3 is tested on C4 Dice and the G1 entropy-gate arm)
        for k in CFG.folds:
            rid = f"seg_f{k}_{init}"
            if manifest.done(rid):
                print(f"[stage 4] {rid} already complete — skipping"); continue
            norm = norms[str(k)]; fdir = cores / f"fold{k}" / init; fdir.mkdir(parents=True, exist_ok=True); t_fold = time.time()
            train_seg = lab[(lab.fold != k) & lab.has_masks].reset_index(drop=True)          # segmentation training: mask-bearing videos only
            train_all = lab[lab.fold != k].reset_index(drop=True); test_df = lab[lab.fold == k].reset_index(drop=True)   # diagnostic features: every labelled video
            # --- inner cross-fit: out-of-sample masks for fitting the quality estimator (nothing else) -------------------
            q_rows, q_ff, q_mf = None, None, None
            if do_inner:
                parts = []
                for j, (tr, te) in enumerate(GroupKFold(n_splits=CFG.inner_splits).split(train_seg, groups=train_seg.group)):
                    m_inner = train_segmenter(train_seg.iloc[tr], cache, norm, init, fdir / f"ckpt_inner{j}.pt", CFG.seg_epochs, log, f"{rid}_inner{j}")
                    parts.append(infer_videos(m_inner, train_seg.iloc[te].reset_index(drop=True), cache, norm, None, "inner")[:3]); del m_inner
                q_rows = pd.concat([p[0] for p in parts], ignore_index=True); q_ff = np.concatenate([p[1] for p in parts]); q_mf = np.concatenate([p[2] for p in parts])
            # --- final segmenter (= C4) and features for ALL fold-k videos from this single frozen core (A1/A2) -----------
            final = train_segmenter(train_seg, cache, norm, init, fdir / "ckpt_final.pt", CFG.seg_epochs, log, f"{rid}_final")
            torch.save(final.state_dict(), fdir / "core_final.pt")
            allv_df = pd.concat([train_all, test_df], ignore_index=True)
            rows, ff, mf, vr, cpu_per_clip = infer_videos(final, allv_df, cache, norm, fdir, "final")
            vr["split"] = np.where(vr.fold == k, "test", "train"); assert (vr.source_pred == "final").all()
            n_tr_feat = int((vr.split == "train").sum()); assert n_tr_feat == len(train_all), "diagnostic features missing for some training videos"
            # --- quality estimators fitted on OOS rows only; the MLP head is the deployable alternative (B4) --------------
            est_info = dict(quality_feat_cpu_ms_per_clip=cpu_per_clip * 1e3)
            if do_inner:
                fit_sel = (q_rows.annotated & q_rows.dice.notna()).to_numpy(); fr_ = q_rows[fit_sel].reset_index(drop=True); ff_, mf_ = q_ff[fit_sel], q_mf[fit_sel]
                vids = fr_.id.unique(); rng = np.random.default_rng(0); hold = set(rng.choice(vids, size=max(1, len(vids) // 5), replace=False)); hs = fr_.id.isin(hold).to_numpy()
                cands = {}
                for cls in (QualityEstimator, lambda: QualityHeadMLP(mf_.shape[1])):
                    e = cls().fit(fr_[~hs].reset_index(drop=True), ff_[~hs], mf_[~hs]); rho_sel = _spear(e.predict(fr_[hs].reset_index(drop=True), ff_[hs], mf_[hs]), fr_[hs].dice)
                    cands[e.name] = dict(rho_oos=rho_sel, est=cls().fit(fr_, ff_, mf_))
                best_rho = max(v["rho_oos"] for v in cands.values() if np.isfinite(v["rho_oos"])) if any(np.isfinite(v["rho_oos"]) for v in cands.values()) else np.nan
                choice = "mlp_head" if (cpu_per_clip * 1e3 > 100 and np.isfinite(cands["mlp_head"]["rho_oos"]) and cands["mlp_head"]["rho_oos"] >= best_rho - 0.05) else "hgb_morphology"
                qe = cands[choice]["est"]; pickle.dump(qe, open(fdir / "quality_estimator.pkl", "wb")); pickle.dump(cands["hgb_morphology"]["est"], open(fdir / "quality_estimator_hgb.pkl", "wb"))
                rows["q_hat"] = qe.predict(rows, ff, mf)
                ann = rows[rows.id.isin(test_df.id) & rows.annotated & rows.dice.notna()]
                for nm, v in cands.items():
                    est_info[f"rho_test_{nm}"] = _spear(v["est"].predict(ann, ff[ann.index.to_numpy()], mf[ann.index.to_numpy()]), ann.dice); est_info[f"rho_oos_{nm}"] = v["rho_oos"]
                est_info.update(quality_estimator_choice=choice, quality_rho=_spear(ann.q_hat, ann.dice), quality_mae=float(np.abs(ann.q_hat - ann.dice).mean()),
                                **{f"rho_{CHAMBERS[c-1]}": _spear(ann[ann.chamber == c].q_hat, ann[ann.chamber == c].dice) for c in range(1, N_CLS)})
                (fdir / "quality_estimator_choice.json").write_text(json.dumps({k_: (float(v) if isinstance(v, (float, np.floating)) else v) for k_, v in est_info.items()}, indent=1))
            else:
                rows["q_hat"] = np.nan; ann = rows[rows.id.isin(test_df.id) & rows.annotated & rows.dice.notna()]
            # --- C4 endpoints on originally annotated frames of the test fold; orientation audit (A3) ---------------------
            seg_metrics = {}
            for c in range(1, N_CLS):
                a = ann[ann.chamber == c]; seg_metrics[f"dice_{CHAMBERS[c-1]}"] = float(a.dice.mean()); seg_metrics[f"msd_px_{CHAMBERS[c-1]}"] = float(a.msd.mean()); seg_metrics[f"hd95_px_{CHAMBERS[c-1]}"] = float(a.hd95.mean())
            ok = vr.dropna(subset=["lv_cx", "rv_cx"]); lv_right_frac = float((ok.lv_cx > ok.rv_cx).mean()) if len(ok) else np.nan
            log.append(f"{rid}_eval", dict(stage=4, arm="C4", init=init, fold=k, **est_info, **seg_metrics, lv_right_of_rv_frac=lv_right_frac, n_train_videos=len(train_all), n_train_videos_with_features=n_tr_feat,
                                           n_train_with_masks=len(train_seg), sec_fold=time.time() - t_fold))
            rows.to_csv(fdir / "quality_rows.csv", index=False)
            # --- cached diagnostic features (every row from the final core) -----------------------------------------------
            qv = rows.groupby(["id", "chamber"]).q_hat.mean().unstack(); F_ = {k_: [] for k_ in ("g", "z", "morph", "ent", "dice", "q")}
            for i in vr.id:
                p = np.load(fdir / "preds" / f"{i}.npz")
                for k_ in ("g", "z", "morph", "ent", "dice"): F_[k_].append(p[k_])
                F_["q"].append(qv.loc[i, list(range(1, N_CLS))].to_numpy(np.float32))
            np.savez_compressed(feats_out / f"{init}_fold{k}.npz", id=vr.id.to_numpy(str), group=vr.group.to_numpy(str), split=vr.split.to_numpy(str), source_pred=vr.source_pred.to_numpy(str),
                                has_masks=vr.has_masks.to_numpy(bool), y_pah=vr.y_pah.to_numpy(np.float32), y_asd=vr.y_asd.to_numpy(np.float32), **{k_: np.stack(v).astype(np.float32) for k_, v in F_.items()})
            manifest.mark(rid)
            print(f"[stage 4] fold {k} {init}: C4 Dice " + " ".join(f"{CHAMBERS[c-1]}={seg_metrics[f'dice_{CHAMBERS[c-1]}']:.3f}" for c in range(1, N_CLS)) +
                  f" | features {n_tr_feat}/{len(train_all)} train + {len(test_df)} test videos, all from the final core" +
                  (f" | quality ρ={est_info['quality_rho']:.3f} ({est_info['quality_estimator_choice']}, CPU {est_info['quality_feat_cpu_ms_per_clip']:.0f} ms/clip)" if do_inner else " | no quality estimator (D0 final-only)") +
                  f" | LV right of RV in {lv_right_frac:.0%} of videos | {(time.time()-t_fold)/60:.1f} min")
    write_version(CFG.ds_cores, dict(folds=CFG.folds, inits=CFG.inits, d0_full=CFG.d0_full)); write_version(CFG.ds_feats, dict(folds=CFG.folds, inits=CFG.inits, features_source="final core for all videos", note="training-video q_hat mildly optimistic (final core saw their masks)"))


if 4 in CFG.stages:
    stage4_segmentation()

[stage 4] depends on: ['cardiacnet-cache-v1', 'student-init-v1']
   cardiacnet-cache-v1    -> k1_work/cardiacnet-cache-v1  (2026-09-29T13:37:36)
   student-init-v1        -> k1_work/student-init-v1  (2026-09-29T13:38:27)


[stage 4] fold 0 D1: C4 Dice LV=0.907 RV=0.865 LA=0.854 RA=0.710 | features 50/50 train + 13 test videos, all from the final core | quality ρ=0.704 (hgb_morphology, CPU 38 ms/clip) | LV right of RV in 100% of videos | 1.9 min


[stage 4] fold 1 D1: C4 Dice LV=0.895 RV=0.870 LA=0.839 RA=0.619 | features 50/50 train + 13 test videos, all from the final core | quality ρ=0.624 (hgb_morphology, CPU 39 ms/clip) | LV right of RV in 100% of videos | 1.8 min


[stage 4] fold 0 D0: C4 Dice LV=0.806 RV=0.622 LA=0.619 RA=0.318 | features 50/50 train + 13 test videos, all from the final core | no quality estimator (D0 final-only) | LV right of RV in 87% of videos | 0.6 min


[stage 4] fold 1 D0: C4 Dice LV=0.666 RV=0.555 LA=0.466 RA=0.254 | features 50/50 train + 13 test videos, all from the final core | no quality estimator (D0 final-only) | LV right of RV in 59% of videos | 0.7 min


In [9]:
# ---------------------------------------------------------------- Cell 9: Stage 5 — diagnostic stage on cached features (S6)
def load_feats(init: str, k: int) -> dict:
    z = np.load(stage_dir(CFG.ds_feats) / f"{init}_fold{k}.npz", allow_pickle=False)
    return {key: z[key] for key in z.files}


def subset(Fd: dict, sel: np.ndarray) -> dict:
    return {k: v[sel] for k, v in Fd.items()}


def make_gate(arm: str, Fd: dict, rng: np.random.Generator) -> np.ndarray:
    """Gate per chamber in [0,1]: G1 1−entropy, G2 estimated quality, G3 quality shuffled across videos, G4 true Dice (oracle ceiling).
    G4 falls back to q̂ where a video has no manual masks (Dice undefined); the fraction of true-oracle entries is reported (A2)."""
    if arm == "G0": return np.zeros_like(Fd["q"])
    if arm == "G1": return np.clip(1.0 - Fd["ent"], 0, 1)
    if arm == "G2": return Fd["q"]
    if arm == "G3": return Fd["q"][rng.permutation(len(Fd["q"]))]
    if arm == "G4": return np.where(np.isnan(Fd["dice"]), Fd["q"], Fd["dice"])
    raise ValueError(arm)


def oracle_fraction(Fd: dict) -> float:
    return float(np.mean(~np.isnan(Fd["dice"]).any(1)))


class Standardiser:
    def __init__(self, Fd):
        self.stats = {k: (Fd[k].mean(0, keepdims=True), Fd[k].std(0, keepdims=True) + 1e-6) for k in ("g", "z", "morph")}
    def __call__(self, Fd):
        return {k: ((Fd[k] - self.stats[k][0]) / self.stats[k][1] if k in self.stats else Fd[k]) for k in Fd}


def split_train_val(Fd: dict, seed: int):
    """Group-wise validation split of the training fold (model selection and calibration never touch the test fold)."""
    tr, va = next(GroupShuffleSplit(n_splits=1, test_size=CFG.val_frac, random_state=seed).split(Fd["id"], groups=Fd["group"]))
    return subset(Fd, tr), subset(Fd, va)


def masked_bce(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    known = ~torch.isnan(y)
    if known.sum() == 0: return logits.sum() * 0.0
    return F.binary_cross_entropy_with_logits(logits[known], y[known])


def _tensors(Fd, gate, device):
    return (torch.tensor(Fd["g"], device=device), torch.tensor(Fd["z"], device=device), torch.tensor(Fd["morph"], device=device),
            torch.tensor(gate, dtype=torch.float32, device=device), torch.tensor(np.stack([Fd["y_pah"], Fd["y_asd"]], 1), device=device))


def train_diag(arm: str, F_tr: dict, F_va: dict, F_te: dict, seed: int, epochs: int, tasks=(0, 1), edl=False, patience=15) -> dict:
    """Adapters, gate relevance and heads only (the core is frozen: everything arrives as cached features). Early stopping on
    validation NLL of the trained tasks with validation PAH AUROC as tie-break (B7); both are logged."""
    seed_everything(seed); rng = np.random.default_rng(seed)
    gates = [make_gate(arm, Fd, rng) for Fd in (F_tr, F_va, F_te)]
    net = DiagNet(arm); model = (EDLHead(net) if edl else net).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr_head, weight_decay=1e-4)
    T = [_tensors(Fd, gt, DEVICE) for Fd, gt in zip((F_tr, F_va, F_te), gates)]
    n = len(F_tr["id"]); bs = min(64, n); best, best_state, best_ep, bad = (-np.inf, -np.inf), None, 0, 0
    def evaluate(t):
        model.eval()
        with torch.no_grad():
            out = model(*t[:4])
        return out
    for ep in range(epochs):
        model.train(); perm = torch.randperm(n, device=DEVICE)
        for b0 in range(0, n, bs):
            i = perm[b0:b0 + bs]; g, z, mo, ga, y = (t[i] for t in T[0])
            out = model(g, z, mo, ga)
            if edl:
                yk = y[:, 0]; k_ = ~torch.isnan(yk); loss = edl_loss(out[k_], yk[k_], lam=min(1.0, ep / max(1, epochs // 2)))
            else:
                loss = sum(masked_bce(out[:, d], y[:, d]) for d in tasks) / len(tasks)
            opt.zero_grad(); loss.backward(); opt.step()
        out_va = evaluate(T[1]); y_va = T[1][4]; y0 = y_va[:, 0].cpu().numpy(); k0 = ~np.isnan(y0)
        if edl:
            p_va = (out_va[:, 1] / out_va.sum(1)).cpu().numpy(); auc = _safe_auc(roc_auc_score, y0[k0], p_va[k0]) if k0.any() else np.nan
            nll = float(-np.mean(np.log(np.clip(np.where(y0[k0] == 1, p_va[k0], 1 - p_va[k0]), 1e-6, 1)))) if k0.any() else float(ep)
        else:
            nll = sum(masked_bce(out_va[:, d], y_va[:, d]) for d in tasks).item() / len(tasks)
            auc = _safe_auc(roc_auc_score, y0[k0], out_va[k0, 0].cpu().numpy()) if (0 in tasks and k0.any()) else np.nan
        score = (-nll, auc if np.isfinite(auc) else -np.inf)                 # NLL primary, AUROC tie-break
        if score > best:
            best, best_state, best_ep, bad = score, copy.deepcopy(model.state_dict()), ep + 1, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    return dict(model=model, val_out=evaluate(T[1]).cpu().numpy(), test_out=evaluate(T[2]).cpu().numpy(), best_epoch=best_ep, best_val_nll=-best[0], best_val_auroc=best[1],
                gates=gates, params=count_params(model))


def stage5_diag_cached():
    require_inputs(5)
    out = stage_dir(CFG.ds_diag, create=True); log = RunLog(out); manifest = RunManifest(out / "runs_manifest.json")
    for init in CFG.inits:
        arms = ["G0", "G1", "G2", "G3", "G4"] if init == "D1" else (["G1", "G2"] if CFG.d0_full else ["G1"])   # D0 (B1): H3 on the entropy-gate arm (no quality estimator needed)
        for k in CFG.folds:
            Fd = load_feats(init, k); assert (Fd["source_pred"] == "final").all(), "diagnostic features must all come from the final core (A1)"
            std = Standardiser(subset(Fd, Fd["split"] == "train")); Fd = std(Fd)
            F_trall, F_te = subset(Fd, Fd["split"] == "train"), subset(Fd, Fd["split"] == "test")
            for seed in CFG.seeds:
                F_tr, F_va = split_train_val(F_trall, seed)
                for arm in arms:
                    rid = f"diag_{init}_{arm}_f{k}_s{seed}"
                    if manifest.done(rid): continue
                    reset_peak_mem()
                    with Timer() as tm:
                        res = train_diag(arm, F_tr, F_va, F_te, seed, CFG.diag_epochs)
                    y = F_te["y_pah"]; kn = ~np.isnan(y); p = expit(res["test_out"][kn, 0])
                    met = binary_metrics(y[kn], p) if kn.sum() else {}
                    ya = F_te["y_asd"]; ka = ~np.isnan(ya); auc_asd = _safe_auc(roc_auc_score, ya[ka], res["test_out"][ka, 1]) if ka.sum() else np.nan
                    np.savez_compressed(out / f"preds_{rid}.npz", id=F_te["id"], group=F_te["group"], y_pah=y, y_asd=ya, test_logits=res["test_out"],
                                        val_id=F_va["id"], val_group=F_va["group"], val_y_pah=F_va["y_pah"], val_y_asd=F_va["y_asd"], val_logits=res["val_out"],
                                        gate_test=res["gates"][2], std_mean_g=std.stats["g"][0], std_std_g=std.stats["g"][1])
                    torch.save(res["model"].state_dict(), out / f"model_{rid}.pt")
                    log.append(rid, dict(stage=5, init=init, arm=arm, fold=k, seed=seed, **{f"{m}_pah": v for m, v in met.items()}, auroc_asd=auc_asd,
                                         best_epoch=res["best_epoch"], best_val_nll=res["best_val_nll"], best_val_auroc=res["best_val_auroc"], params_trainable=res["params"],
                                         g4_oracle_frac_test=(oracle_fraction(F_te) if arm == "G4" else np.nan), g4_oracle_frac_train=(oracle_fraction(F_tr) if arm == "G4" else np.nan),
                                         n_test_with_masks=int(F_te["has_masks"].sum()), sec=tm.dt, peak_mem_mb=peak_mem_mb()))
                    manifest.mark(rid)
                    print(f"[stage 5] {rid}: AUROC {met.get('auroc', np.nan):.3f} AUPRC {met.get('auprc', np.nan):.3f} ASD {auc_asd:.3f} ({tm.dt:.0f}s, best ep {res['best_epoch']})")
    write_version(CFG.ds_diag, dict(folds=CFG.folds, seeds=CFG.seeds, inits=CFG.inits))
    df = log.read()
    if len(df):
        print(df.groupby(["init", "arm"]).auroc_pah.agg(["mean", "std", "count"]).round(3))


if 5 in CFG.stages:
    stage5_diag_cached()

[stage 5] depends on: ['cardiacnet-cache-v1', 'diag-features-v1']
   cardiacnet-cache-v1    -> k1_work/cardiacnet-cache-v1  (2026-09-29T13:37:36)
   diag-features-v1       -> k1_work/diag-features-v1  (2026-09-29T13:43:26)


[stage 5] diag_D1_G0_f0_s0: AUROC 1.000 AUPRC 1.000 ASD 0.667 (0s, best ep 1)
[stage 5] diag_D1_G1_f0_s0: AUROC 0.556 AUPRC 0.766 ASD 1.000 (0s, best ep 1)


[stage 5] diag_D1_G2_f0_s0: AUROC 0.444 AUPRC 0.729 ASD 1.000 (0s, best ep 1)
[stage 5] diag_D1_G3_f0_s0: AUROC 0.444 AUPRC 0.729 ASD 1.000 (0s, best ep 1)


[stage 5] diag_D1_G4_f0_s0: AUROC 0.389 AUPRC 0.704 ASD 1.000 (0s, best ep 1)
[stage 5] diag_D1_G0_f0_s1: AUROC 0.889 AUPRC 0.948 ASD 1.000 (0s, best ep 26)


[stage 5] diag_D1_G1_f0_s1: AUROC 0.889 AUPRC 0.948 ASD 1.000 (0s, best ep 27)


...

[stage 5] diag_D1_G3_f1_s2: AUROC 1.000 AUPRC 1.000 ASD 1.000 (0s, best ep 10)
[stage 5] diag_D1_G4_f1_s2: AUROC 1.000 AUPRC 1.000 ASD 1.000 (0s, best ep 10)


[stage 5] diag_D0_G1_f0_s0: AUROC 0.722 AUPRC 0.818 ASD 1.000 (0s, best ep 8)
[stage 5] diag_D0_G1_f0_s1: AUROC 0.778 AUPRC 0.897 ASD 1.000 (0s, best ep 30)


[stage 5] diag_D0_G1_f0_s2: AUROC 1.000 AUPRC 1.000 ASD 1.000 (0s, best ep 30)
[stage 5] diag_D0_G1_f1_s0: AUROC 0.458 AUPRC 0.593 ASD 1.000 (0s, best ep 1)


[stage 5] diag_D0_G1_f1_s1: AUROC 0.958 AUPRC 0.976 ASD 1.000 (0s, best ep 12)
[stage 5] diag_D0_G1_f1_s2: AUROC 0.958 AUPRC 0.976 ASD 1.000 (0s, best ep 19)


           mean    std  count
init arm                     
D0   G1   0.812  0.206      6
D1   G0   0.873  0.210      6
     G1   0.806  0.222      6
     G2   0.836  0.207      6
     G3   0.836  0.207      6
     G4   0.826  0.228      6


In [10]:
# ---------------------------------------------------------------- Cell 10: Stage 6 — joint training with CAGrad, input ablations, baselines (S7–S8)
class JointModel(nn.Module):
    """G5: trainable core + FPN + G2-style diagnostic head. Anatomy tokens use stop-gradient soft masks. The gate is the SAME fixed
    per-video q̂ that G2 uses (stage-4 D1 final-core estimate from diag-features-v1), so G2 vs G5 isolates freezing vs joint training (A5).
    No estimator runs inside the training loop."""
    def __init__(self, core: SegModel):
        super().__init__(); self.core = core; self.diag = DiagNet("G2"); self.ln_g, self.ln_z = nn.LayerNorm(D_FEAT), nn.LayerNorm(D_FEAT)
    def forward(self, x, q):
        logits, f4, _ = self.core(x); prob = logits[-1].float().softmax(1)
        p_small = F.adaptive_avg_pool3d(prob.detach(), f4.shape[-3:])
        z = torch.einsum("bcthw,bdthw->bcd", p_small, f4.float()) / (p_small.sum(dim=(2, 3, 4))[..., None] + 1e-6)
        g = f4.float().mean(dim=(2, 3, 4))
        with torch.no_grad():                                    # morphology scalars from the model's own argmax masks (cheap, no gradient)
            pred = prob.argmax(1); B, T = pred.shape[:2]
            areas = torch.stack([(pred == c).float().mean(dim=(2, 3)) for c in range(N_CLS)], 2)      # [B,T,C]
            ed = areas[:, :, 1].argmax(1); ar = areas[torch.arange(B), ed]
            morph = torch.stack([torch.clamp(ar[:, 2] / (ar[:, 1] + 1e-4), max=5.0), ar[:, 4] * 10.0, (areas[:, :, 2].amax(1) - areas[:, :, 2].amin(1)) / (areas[:, :, 2].amax(1) + 1e-6)], 1)
        return logits, self.diag(self.ln_g(g), self.ln_z(z[:, 1:]), morph, q)


class QLookup:
    """Per-video fixed quality vectors from diag-features-v1 (D1, fold k) for the joint arm."""
    def __init__(self, k: int):
        Fd = load_feats("D1", k); self.q = {i: torch.tensor(q) for i, q in zip(Fd["id"], Fd["q"])}
    def __call__(self, ids, device): return torch.stack([self.q[i] for i in ids]).to(device)


def cagrad_step(shared: List[torch.Tensor], losses: List[torch.Tensor], opt, c=0.5) -> float:
    """Conflict-averse gradient descent (Liu et al. 2021) for two tasks on the shared parameters; task-specific parameters keep
    their own gradients. Returns the cosine between the two task gradients (logged per epoch)."""
    G = []
    for L in losses:
        gr = torch.autograd.grad(L, shared, retain_graph=True, allow_unused=True)
        G.append(torch.cat([(g if g is not None else torch.zeros_like(p)).flatten() for g, p in zip(gr, shared)]))
    opt.zero_grad(set_to_none=True); sum(losses).backward()
    g1, g2 = G; g0 = (g1 + g2) / 2; phi = c ** 2 * (g0 @ g0)
    def obj(w):
        gw = w * g1 + (1 - w) * g2; return float(gw @ g0 + phi.sqrt() * gw.norm())
    w = minimize_scalar(obj, bounds=(0.0, 1.0), method="bounded").x
    gw = w * g1 + (1 - w) * g2; d = (g0 + phi.sqrt() / (gw.norm() + 1e-12) * gw) / (1 + c ** 2)
    off = 0
    for p in shared:
        n = p.numel(); p.grad = d[off:off + n].view_as(p).clone(); off += n
    opt.step()
    return float((g1 @ g2) / (g1.norm() * g2.norm() + 1e-12))


def eval_clip_model(model, df, cache, norm, channels="diff", joint=False, qlk: Optional["QLookup"] = None):
    """Test-time pass for the raw-video arms: PAH/ASD logits per video and, for the joint model, RV Dice on annotated frames."""
    model.eval(); ds = ClipDataset(df, cache, norm, channels=channels)
    dl = torch.utils.data.DataLoader(ds, batch_size=CFG.batch, shuffle=False, num_workers=CFG.num_workers); outs, dices = [], []
    with torch.no_grad():
        for x, m, _, idx in dl:
            with autocast_ctx():
                o = model(x.to(DEVICE), qlk(df.id.iloc[idx.numpy()].tolist(), DEVICE)) if joint else model(x.to(DEVICE))
            if joint:
                logits, o = o; pred = logits[-1].argmax(1).cpu().numpy(); mm = m.numpy()
                for b in range(len(pred)):
                    for t in range(pred.shape[1]):
                        if (mm[b, t] != IGNORE).any(): dices.append([dice_binary(pred[b, t] == c, mm[b, t] == c) for c in range(1, N_CLS)])
            outs.append(o.float().cpu().numpy())
    return np.concatenate(outs), (np.array(dices).mean(0) if dices else np.full(4, np.nan))


def train_raw_arm(arm: str, k: int, seed: int, cache, man, norms, out: Path, log: RunLog, joint=False):
    """Shared trainer for G5 (joint + CAGrad) and the end-to-end screening arms A0/A7/A8/C1/C2, with group-wise validation split,
    early stopping on validation NLL with PAH AUROC as tie-break (the same rule as the cached-feature arms), per-epoch checkpoints and resume."""
    norm = norms[str(k)]; lab = man[man.source.isin(["PAH", "ASD"])].reset_index(drop=True)
    trall = lab[lab.fold != k].reset_index(drop=True); te = lab[lab.fold == k].reset_index(drop=True)
    tr_i, va_i = next(GroupShuffleSplit(n_splits=1, test_size=CFG.val_frac, random_state=seed).split(trall, groups=trall.group))
    tr, va = trall.iloc[tr_i].reset_index(drop=True), trall.iloc[va_i].reset_index(drop=True)
    seed_everything(seed); channels = {"A7": "intensity", "A8": "raw2"}.get(arm, "diff")
    qlk = None
    if joint:
        qlk = QLookup(k); net = JointModel(init_core("D1"))
    elif arm in ("A0", "A7", "A8"): net = ClipClassifier(init_core("D1"), temporal_block=(arm == "A8"))
    elif arm == "C1": net = build_cardnet_x_lite()
    elif arm == "C2": net = r3d18_two_channel()
    else: raise ValueError(arm)
    model = net.to(DEVICE) if joint else parallelise(net)      # CAGrad needs direct access to per-parameter grads: no DataParallel for G5
    epochs = CFG.joint_epochs if joint else CFG.screen_epochs
    dl = torch.utils.data.DataLoader(ClipDataset(tr, cache, norm, channels=channels, augment=True), batch_size=CFG.batch, shuffle=True, drop_last=len(tr) > CFG.batch, num_workers=CFG.num_workers)
    opt = torch.optim.AdamW(model.parameters(), lr=(CFG.lr_core if arm in ("A0", "A7", "A8") or joint else 1e-3), weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs * len(dl))); scaler = make_scaler()
    rid = f"{arm}_f{k}_s{seed}"; ck = out / f"ckpt_{rid}.pt"; start, extra = 0, {"best": (-np.inf, -np.inf), "best_ep": 0, "cos": []}
    if ck.exists(): start, extra = load_ckpt(ck, model, opt, sched)
    best_path = out / f"best_{rid}.pt"; shared = unwrap(model).core.core_parameters() if joint else None
    reset_peak_mem(); t0 = time.time()
    for ep in range(start, epochs):
        model.train(); cos_ep = []
        for x, m, y, idx in dl:
            x, m, y = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)
            if joint:                                            # fp32 for CAGrad (manual gradient surgery is incompatible with GradScaler)
                logits, o = model(x, qlk(tr.id.iloc[idx.numpy()].tolist(), DEVICE)); l_seg, l_diag = seg_loss(logits, m), masked_bce(o[:, 0], y[:, 0]) + masked_bce(o[:, 1], y[:, 1])
                cos_ep.append(cagrad_step(shared, [l_seg, l_diag], opt))
            else:
                with autocast_ctx():
                    o = model(x); loss = masked_bce(o[:, 0], y[:, 0]) + masked_bce(o[:, 1], y[:, 1])
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 3.0); scaler.step(opt); scaler.update()
            sched.step()
        o_va, _ = eval_clip_model(model, va, cache, norm, channels, joint, qlk); kv = ~np.isnan(va.y_pah.to_numpy()); yv = torch.tensor(va[["y_pah", "y_asd"]].to_numpy(np.float32)); ov = torch.tensor(o_va)
        nll = float(masked_bce(ov[:, 0], yv[:, 0]) + masked_bce(ov[:, 1], yv[:, 1])) / 2
        auc = _safe_auc(roc_auc_score, va.y_pah.to_numpy()[kv], o_va[kv, 0]) if kv.sum() else np.nan
        score = (-nll, auc if np.isfinite(auc) else -np.inf)             # NLL primary, AUROC tie-break
        if joint: extra["cos"].append(float(np.mean(cos_ep)))
        if score > tuple(extra["best"]):
            extra["best"], extra["best_ep"] = list(score), ep + 1; torch.save(unwrap(model).state_dict(), best_path)
        save_ckpt(ck, model, opt, sched, ep + 1, extra)
    unwrap(model).load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=False))
    o_te, dice_te = eval_clip_model(model, te, cache, norm, channels, joint, qlk); o_va, _ = eval_clip_model(model, va, cache, norm, channels, joint, qlk)
    y = te.y_pah.to_numpy(); kn = ~np.isnan(y); met = binary_metrics(y[kn], expit(o_te[kn, 0])) if kn.sum() else {}
    ya = te.y_asd.to_numpy(); ka = ~np.isnan(ya); auc_asd = _safe_auc(roc_auc_score, ya[ka], o_te[ka, 1]) if ka.sum() else np.nan
    np.savez_compressed(out / f"preds_{rid}.npz", id=te.id.to_numpy(str), group=te.group.to_numpy(str), y_pah=y, y_asd=ya, test_logits=o_te,
                        val_id=va.id.to_numpy(str), val_group=va.group.to_numpy(str), val_y_pah=va.y_pah.to_numpy(), val_y_asd=va.y_asd.to_numpy(), val_logits=o_va)
    row = dict(stage=6, arm=arm, init=("D1" if arm in ("A0", "A7", "A8") or joint else "scratch"), fold=k, seed=seed, **{f"{m}_pah": v for m, v in met.items()}, auroc_asd=auc_asd,
               best_epoch=extra["best_ep"], best_val_nll=-extra["best"][0], best_val_auroc=extra["best"][1], params=count_params(unwrap(model)), sec=time.time() - t0, peak_mem_mb=peak_mem_mb(), channels=channels,
               kinetics_init=(getattr(unwrap(model), "kinetics_init", np.nan) if arm == "C2" else np.nan), c1_stand_in=(isinstance(unwrap(model), _CaRDNetXLiteStandIn) if arm == "C1" else np.nan))
    if joint:
        row.update({f"dice_{n}": float(d) for n, d in zip(CHAMBERS, dice_te)}, grad_cos_mean=float(np.mean(extra["cos"])), grad_cos_last=extra["cos"][-1], grad_cos_curve=json.dumps(extra["cos"]))
    log.append(rid, row)
    print(f"[stage 6] {rid}: AUROC {met.get('auroc', np.nan):.3f} | params {row['params']/1e6:.2f} M" + (f" | RV Dice {dice_te[1]:.3f} | grad-cos {row['grad_cos_mean']:.2f}" if joint else "") + f" ({(time.time()-t0)/60:.1f} min)")
    return row


def seed_noise(diag_df: pd.DataFrame, arm="G2", init="D1") -> float:
    """Seed noise = std over seeds of the fold-averaged AUROC of the reference arm."""
    d = diag_df[(diag_df.arm == arm) & (diag_df.init == init)]
    return float(d.groupby("seed").auroc_pah.mean().std(ddof=0)) if d.seed.nunique() > 1 else float("nan")


def stage6_joint_and_ablations():
    require_inputs(6)
    cache, man, norms = load_cache()
    joint_out, abl_out = stage_dir(CFG.ds_joint, create=True), stage_dir(CFG.ds_abl, create=True)
    jlog, alog = RunLog(joint_out), RunLog(abl_out); jman, aman = RunManifest(joint_out / "runs_manifest.json"), RunManifest(abl_out / "runs_manifest.json")
    # --- S7: G5 joint + CAGrad at 5 folds × 3 seeds --------------------------------------------------------------
    for k in CFG.folds:
        for seed in CFG.seeds:
            rid = f"G5_f{k}_s{seed}"
            if jman.done(rid): continue
            train_raw_arm("G5", k, seed, cache, man, norms, joint_out, jlog, joint=True); jman.mark(rid)
    # --- S8: screening arms at 5 folds × 1 seed; C4 rows copied from stage 4 -------------------------------------
    for arm in ["A0", "A7", "A8", "C1", "C2"]:
        for k in CFG.folds:
            rid = f"{arm}_f{k}_s0"
            if aman.done(rid): continue
            train_raw_arm(arm, k, 0, cache, man, norms, abl_out, alog); aman.mark(rid)
    seg_df = RunLog(stage_dir(CFG.ds_cores)).read()
    if len(seg_df):
        seg_df[seg_df.arm.eq("C4")].to_csv(abl_out / "C4_from_stage4.csv", index=False)
    # --- promotion rule: screening arms whose effect exceeds seed noise are promoted to seeds 1,2 (S15) --------------
    diag_df, abl_df = RunLog(stage_dir(CFG.ds_diag)).read(), alog.read(); noise = seed_noise(diag_df)
    ref = {"A7": "A0", "A8": "A0", "C1": "G2", "C2": "G2"}; queue = []
    for arm, r in ref.items():
        a = abl_df[(abl_df.arm == arm) & (abl_df.seed == 0)].auroc_pah.mean()
        b = abl_df[(abl_df.arm == r) & (abl_df.seed == 0)].auroc_pah.mean() if r in abl_df.arm.values else diag_df[(diag_df.arm == r) & (diag_df.init == "D1")].auroc_pah.mean()
        eff = float(abs(a - b)); promoted = bool(np.isfinite(noise) and eff > noise)
        queue.append(dict(arm=arm, reference=r, effect=eff, seed_noise=noise, promote=promoted, seeds=[s for s in CFG.seeds if s != 0]))
    (abl_out / "promotion_queue.json").write_text(json.dumps(queue, indent=1))
    print("[stage 6] promotion queue:", [(q["arm"], round(q["effect"], 3), q["promote"]) for q in queue])
    if os.environ.get("K1_PROMOTE", "1" if SMOKE_TEST else "0") == "1":
        for q in queue:
            if not q["promote"]: continue
            for seed in q["seeds"]:
                for k in CFG.folds:
                    rid = f"{q['arm']}_f{k}_s{seed}"
                    if aman.done(rid): continue
                    train_raw_arm(q["arm"], k, seed, cache, man, norms, abl_out, alog); aman.mark(rid)
    write_version(CFG.ds_joint, dict(folds=CFG.folds, seeds=CFG.seeds)); write_version(CFG.ds_abl, dict(folds=CFG.folds, promotion=queue))
    df = pd.concat([jlog.read(), alog.read()], ignore_index=True)
    if len(df): print(df.groupby("arm")[["auroc_pah", "params"]].mean().round(3))


if 6 in CFG.stages:
    stage6_joint_and_ablations()

[stage 6] depends on: ['cardiacnet-cache-v1', 'student-init-v1', 'seg-cores-v1', 'diag-features-v1', 'diag-results-v1']


   cardiacnet-cache-v1    -> k1_work/cardiacnet-cache-v1  (2026-09-29T13:37:36)
   student-init-v1        -> k1_work/student-init-v1  (2026-09-29T13:38:27)
   seg-cores-v1           -> k1_work/seg-cores-v1  (2026-09-29T13:43:26)
   diag-features-v1       -> k1_work/diag-features-v1  (2026-09-29T13:43:26)
   diag-results-v1        -> k1_work/diag-results-v1  (2026-09-29T13:43:33)


[stage 6] G5_f0_s0: AUROC 0.833 | params 3.44 M | RV Dice 0.619 | grad-cos -0.01 (1.1 min)


[stage 6] G5_f0_s1: AUROC 0.889 | params 3.44 M | RV Dice 0.722 | grad-cos 0.02 (1.1 min)


[stage 6] G5_f0_s2: AUROC 0.500 | params 3.44 M | RV Dice 0.618 | grad-cos 0.02 (1.0 min)


...

[stage 6] C1_f1_s2: AUROC 0.833 | params 1.70 M (0.2 min)


[stage 6] C2_f0_s1: AUROC 0.500 | params 33.16 M (0.3 min)


[stage 6] C2_f1_s1: AUROC 0.750 | params 33.16 M (0.3 min)


[stage 6] C2_f0_s2: AUROC 0.833 | params 33.16 M (0.3 min)


[stage 6] C2_f1_s2: AUROC 0.812 | params 33.16 M (0.3 min)
     auroc_pah      params
arm                       
A0       0.319   3421174.0
A7       0.438   3421174.0
A8       0.451   3421180.0
C1       0.481   1697602.0
C2       0.663  33157890.0
G5       0.523   3438788.0


In [11]:
# ---------------------------------------------------------------- Cell 11: Stage 7 — calibration, uncertainty, statistics, withdrawal conditions (S9)
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

SIGMOID = expit
_trapz = getattr(np, "trapezoid", None) or getattr(np, "trapz")


def pred_index() -> pd.DataFrame:
    rows = []
    for ds, pat in ((CFG.ds_diag, r"preds_diag_(?P<init>D\d)_(?P<arm>G\d)_f(?P<fold>\d+)_s(?P<seed>\d+)\.npz"), (CFG.ds_joint, r"preds_(?P<arm>G5)_f(?P<fold>\d+)_s(?P<seed>\d+)\.npz"), (CFG.ds_abl, r"preds_(?P<arm>[AC]\d)_f(?P<fold>\d+)_s(?P<seed>\d+)\.npz")):
        try: d = stage_dir(ds)
        except FileNotFoundError: continue
        for f in d.glob("preds_*.npz"):
            m = re.fullmatch(pat, f.name)
            if m: rows.append(dict(path=str(f), init=m.groupdict().get("init", "D1" if m["arm"] in ("G5", "A0", "A7", "A8") else "scratch"), arm=m["arm"], fold=int(m["fold"]), seed=int(m["seed"])))
    return pd.DataFrame(rows)


def oof(PI: pd.DataFrame, arm: str, init="D1", seed=0, task="pah", split="test") -> Optional[pd.DataFrame]:
    """Pooled out-of-fold predictions (each video predicted by the model that held its fold out); None if any fold is missing."""
    sel = PI[(PI.arm == arm) & (PI.init == init) & (PI.seed == seed)]
    if sel.fold.nunique() < len(CFG.folds): return None
    parts = []
    for _, r in sel.iterrows():
        z = np.load(r.path); pre = "" if split == "test" else "val_"; col = 0 if task == "pah" else 1
        parts.append(pd.DataFrame(dict(id=z[pre + "id"], group=z[pre + "group"], y=z[f"{pre}y_{task}"], logit=z[pre + ("test_logits" if split == "test" else "logits")][:, col], fold=r.fold)))
    df = pd.concat(parts, ignore_index=True); df = df[~np.isnan(df.y)].reset_index(drop=True); df["p"] = SIGMOID(df.logit)
    return df


def ensemble(PI, arm, init="D1", seeds=None, task="pah"):
    dfs = [oof(PI, arm, init, s, task) for s in (seeds or CFG.seeds)]; dfs = [d for d in dfs if d is not None]
    if not dfs: return None, []
    base = dfs[0][["id", "group", "y", "fold"]].copy(); P = np.stack([d.set_index("id").loc[base.id, "p"].to_numpy() for d in dfs], 1)
    base["p"] = P.mean(1); base["p_var"] = P.var(1); base["logit"] = np.log(np.clip(base.p, 1e-6, 1 - 1e-6) / (1 - np.clip(base.p, 1e-6, 1 - 1e-6)))
    return base, dfs


# ---------------- statistics over acquisition groups -----------------------------------------------------------------
def _auc_w(y, p, w): return roc_auc_score(y, p, sample_weight=w) if len(np.unique(y[w > 0])) == 2 else np.nan


def paired_group_bootstrap(A: pd.DataFrame, B: pd.DataFrame, n: int, seed=0):
    """Cluster bootstrap over acquisition groups of the paired ΔAUROC (A − B); 95 % percentile interval."""
    M = A[["id", "group", "y", "p"]].merge(B[["id", "p"]], on="id", suffixes=("_a", "_b")); groups = M.group.unique(); gi = pd.factorize(M.group)[0]
    rng = np.random.default_rng(seed); deltas = []
    for _ in range(n):
        cnt = np.bincount(rng.integers(0, len(groups), len(groups)), minlength=len(groups)); w = cnt[gi].astype(float)
        deltas.append(_auc_w(M.y.to_numpy(), M.p_a.to_numpy(), w) - _auc_w(M.y.to_numpy(), M.p_b.to_numpy(), w))
    deltas = np.array(deltas); deltas = deltas[np.isfinite(deltas)]
    obs = roc_auc_score(M.y, M.p_a) - roc_auc_score(M.y, M.p_b)
    return dict(delta=float(obs), ci_lo=float(np.percentile(deltas, 2.5)), ci_hi=float(np.percentile(deltas, 97.5)), n_groups=int(len(groups)))


def permutation_test(A: pd.DataFrame, B: pd.DataFrame, n: int, seed=0) -> float:
    """Group-level label-swap permutation test of ΔAUROC = 0 (arm identity swapped for all videos of a group at once)."""
    M = A[["id", "group", "y", "p"]].merge(B[["id", "p"]], on="id", suffixes=("_a", "_b")); gi = pd.factorize(M.group)[0]; G = gi.max() + 1
    y, pa, pb = M.y.to_numpy(), M.p_a.to_numpy(), M.p_b.to_numpy(); obs = abs(roc_auc_score(y, pa) - roc_auc_score(y, pb)); rng = np.random.default_rng(seed); hits = 0
    for _ in range(n):
        flip = (rng.random(G) < 0.5)[gi]; a = np.where(flip, pb, pa); b = np.where(flip, pa, pb)
        hits += abs(roc_auc_score(y, a) - roc_auc_score(y, b)) >= obs - 1e-12
    return (hits + 1) / (n + 1)


def seed_fold_interval(runs: pd.DataFrame, a: dict, b: dict, metric="auroc_pah"):
    """Paired difference over matched (fold, seed) runs: mean and 95 % t-interval — the '5-fold × 3-seed interval' of the design."""
    A = runs[np.logical_and.reduce([runs[k] == v for k, v in a.items()])].set_index(["fold", "seed"])[metric]
    B = runs[np.logical_and.reduce([runs[k] == v for k, v in b.items()])].set_index(["fold", "seed"])[metric]
    if len(B.index.unique()) == 1 and len(A.index.unique()) > 1:            # 1-seed screening arm vs 3-seed reference: pair by fold
        A = A.groupby("fold").mean(); B = B.groupby("fold").mean()
    d = (A - B).dropna()
    if len(d) < 2: return dict(mean=float(d.mean()) if len(d) else np.nan, lo=np.nan, hi=np.nan, n=int(len(d)))
    h = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / math.sqrt(len(d))
    return dict(mean=float(d.mean()), lo=float(d.mean() - h), hi=float(d.mean() + h), n=int(len(d)))


# ---------------- calibration, conformal abstention, selective prediction -------------------------------------------------
def fit_temperature(logit, y):
    f = lambda lt: -np.mean(y * np.log(SIGMOID(logit / np.exp(lt)) + 1e-9) + (1 - y) * np.log(1 - SIGMOID(logit / np.exp(lt)) + 1e-9))
    return float(np.exp(minimize_scalar(f, bounds=(-3, 3), method="bounded").x))


def fit_platt(logit, y):
    from sklearn.linear_model import LogisticRegression
    m = LogisticRegression(C=1e4, max_iter=1000).fit(logit[:, None], y); return float(m.coef_[0, 0]), float(m.intercept_[0])


def calibrated_oof(PI, arm, init, seed, method):
    """Fit the calibrator on each fold's validation split and apply it to that fold's test logits."""
    te, va = oof(PI, arm, init, seed), oof(PI, arm, init, seed, split="val"); out = []
    for k, t in te.groupby("fold"):
        v = va[va.fold == k]; t = t.copy()
        if len(np.unique(v.y)) < 2: out.append(t); continue
        if method == "temperature": t["p"] = SIGMOID(t.logit / fit_temperature(v.logit.to_numpy(), v.y.to_numpy()))
        elif method == "platt": a, b = fit_platt(v.logit.to_numpy(), v.y.to_numpy()); t["p"] = SIGMOID(a * t.logit + b)
        out.append(t)
    return pd.concat(out, ignore_index=True)


def conformal_abstention(PI, arm, init, seed, alpha):
    """Split-conformal prediction sets {0},{1},{0,1} with score 1 − p̂(y); calibration on the validation split per fold."""
    te, va = oof(PI, arm, init, seed), oof(PI, arm, init, seed, split="val"); cov, abst, acc_single, n = 0, 0, 0, 0
    for k, t in te.groupby("fold"):
        v = va[va.fold == k]; s = np.where(v.y == 1, 1 - v.p, v.p); m = len(s)
        q = np.quantile(s, min(1.0, math.ceil((m + 1) * (1 - alpha)) / m), method="higher") if m else 1.0
        in1, in0 = (1 - t.p) <= q, t.p <= q; size = in1.astype(int) + in0.astype(int)
        cov += int(np.sum(np.where(t.y == 1, in1, in0))); abst += int(np.sum(size != 1)); n += len(t)
        single = size == 1; acc_single += int(np.sum((in1[single] & (t.y[single] == 1)) | (in0[single] & (t.y[single] == 0))))
    return dict(coverage=cov / n, abstention=abst / n, acc_on_answered=acc_single / max(1, n - abst))


def selective_metrics(conf, err, coverage=0.9):
    """AURC, AUGRC (generalised risk = P(error ∧ selected)) and accuracy at a fixed coverage."""
    o = np.argsort(-conf); e = np.asarray(err, float)[o]; n = len(e); cum = np.cumsum(e); cov = np.arange(1, n + 1) / n
    risk = cum / np.arange(1, n + 1); grisk = cum / n
    return dict(aurc=float(_trapz(risk, cov)), augrc=float(_trapz(grisk, cov)), acc_at_cov=float(1 - risk[min(n - 1, int(math.ceil(coverage * n)) - 1)]))


def sens_threshold(PI, arm, init, seed, target):
    va = oof(PI, arm, init, seed, split="val"); fpr, tpr, thr = roc_curve(va.y, va.p); ok = np.where(tpr >= target)[0]
    return float(thr[ok[0]]) if len(ok) else 0.5


def stage7_statistics():
    require_inputs(7)
    out = stage_dir(CFG.ds_final, create=True); PI = pred_index(); summary, findings = {}, {}
    diag_runs = RunLog(stage_dir(CFG.ds_diag)).read(); seg_runs = RunLog(stage_dir(CFG.ds_cores)).read()
    joint_runs = RunLog(stage_dir(CFG.ds_joint)).read() if (CFG.work / CFG.ds_joint).exists() or (CFG.kaggle_input / CFG.ds_joint).exists() else pd.DataFrame()
    abl_runs = RunLog(stage_dir(CFG.ds_abl)).read() if (CFG.work / CFG.ds_abl).exists() or (CFG.kaggle_input / CFG.ds_abl).exists() else pd.DataFrame()
    runs = pd.concat([diag_runs, joint_runs, abl_runs], ignore_index=True)
    # ---- ensembles and primary endpoints per arm --------------------------------------------------------------------
    arms = [("D1", a) for a in ["G0", "G1", "G2", "G3", "G4", "G5", "A0", "A7", "A8"]] + [("D0", "G1"), ("D0", "G2"), ("scratch", "C1"), ("scratch", "C2")]
    n_masks = {}
    for k in CFG.folds:
        Fd = load_feats("D1", k); te_ = subset(Fd, Fd["split"] == "test"); n_masks[k] = int(te_["has_masks"].sum())
    n_test_with_masks = int(sum(n_masks.values()))
    ens, table = {}, []
    for init, arm in arms:
        E, per_seed = ensemble(PI, arm, init)
        if E is None: continue
        ens[(init, arm)] = E; m = binary_metrics(E.y, E.p)
        table.append(dict(init=init, arm=arm, n_seeds=len(per_seed), **{f"ens_{k}": v for k, v in m.items()}, seed_mean_auroc=float(np.mean([roc_auc_score(d.y, d.p) for d in per_seed])),
                          n_true_oracle=(n_test_with_masks if arm == "G4" else np.nan)))
    table = pd.DataFrame(table); table.to_csv(out / "arm_table.csv", index=False); print(table.round(3).to_string(index=False))
    G2 = ens[("D1", "G2")]; noise = seed_noise(diag_runs); summary["seed_noise_auroc_G2"] = noise
    # ---- confirmatory family: G2 vs G1, G2 vs G3, G2 vs G0, D1 vs D0 (Holm-corrected) -------------------------------
    conf_pairs = [(("D1", "G2"), ("D1", "G1")), (("D1", "G2"), ("D1", "G3")), (("D1", "G2"), ("D1", "G0")), (("D1", "G1"), ("D0", "G1"))] + ([(("D1", "G2"), ("D0", "G2"))] if CFG.d0_full else [])   # H3 on the entropy-gate arm (B1)
    tests = []
    for a, b in conf_pairs:
        if a not in ens or b not in ens: continue
        bs = paired_group_bootstrap(ens[a], ens[b], CFG.n_boot); p = permutation_test(ens[a], ens[b], CFG.n_perm)
        iv = seed_fold_interval(runs, dict(init=a[0], arm=a[1]), dict(init=b[0], arm=b[1]))
        tests.append(dict(comparison=f"{a[0]}:{a[1]} − {b[0]}:{b[1]}", **{f"boot_{k}": v for k, v in bs.items()}, perm_p=p, **{f"seedfold_{k}": v for k, v in iv.items()}))
    if tests:
        T_ = pd.DataFrame(tests); T_["perm_p_holm"] = holm(T_.perm_p.tolist()); T_.to_csv(out / "confirmatory_tests.csv", index=False); print("\n", T_.round(4).to_string(index=False))
        summary["confirmatory"] = T_.to_dict("records")
    # ---- screening family (1 seed): effect vs seed noise --------------------------------------------------------------
    screen = []
    for a, b in [(("D1", "A0"), ("D1", "A7")), (("D1", "A0"), ("D1", "A8")), (("D1", "G2"), ("scratch", "C1")), (("D1", "G2"), ("scratch", "C2")), (("D1", "G2"), ("D1", "G5"))]:
        if a in ens and b in ens:
            iv = seed_fold_interval(runs, dict(init=a[0], arm=a[1]), dict(init=b[0], arm=b[1]))
            screen.append(dict(comparison=f"{a[0]}:{a[1]} − {b[0]}:{b[1]}", **iv, exceeds_seed_noise=bool(np.isfinite(noise) and abs(iv["mean"]) > noise)))
    if screen: pd.DataFrame(screen).to_csv(out / "screening_tests.csv", index=False); summary["screening"] = screen
    # ---- calibration: temperature vs Platt; conformal abstention; sensitivity-anchored operating point --------------------
    cal = []
    for method in ("none", "temperature", "platt"):
        ms = [binary_metrics(d.y, d.p) for d in (oof(PI, "G2", "D1", s) if method == "none" else calibrated_oof(PI, "G2", "D1", s, method) for s in CFG.seeds) if d is not None]
        cal.append(dict(method=method, **{k: float(np.mean([m[k] for m in ms])) for k in ("nll", "ece", "brier", "cal_slope", "auroc")}))
    cal = pd.DataFrame(cal); cal.to_csv(out / "calibration.csv", index=False); print("\n", cal.round(4).to_string(index=False))
    conf = pd.DataFrame([conformal_abstention(PI, "G2", "D1", s, 1 - CFG.coverage) for s in CFG.seeds]).mean().to_dict(); summary["conformal_G2"] = conf
    thr = float(np.mean([sens_threshold(PI, "G2", "D1", s, CFG.target_sensitivity) for s in CFG.seeds])); yhat = (G2.p >= thr).astype(int)
    tp, tn, fp, fn = int(((yhat == 1) & (G2.y == 1)).sum()), int(((yhat == 0) & (G2.y == 0)).sum()), int(((yhat == 1) & (G2.y == 0)).sum()), int(((yhat == 0) & (G2.y == 1)).sum())
    summary["operating_point_G2"] = dict(threshold=thr, sensitivity=tp / max(1, tp + fn), specificity=tn / max(1, tn + fp), npv=tn / max(1, tn + fn))
    # ---- uncertainty U1–U4 inside G2: MSR, ensemble, MC dropout, EDL --------------------------------------------------------
    err = (G2.p > 0.5).astype(int) != G2.y.astype(int); G2s0 = oof(PI, "G2", "D1", CFG.seeds[0]).set_index("id").loc[G2.id]
    unc = {"U0_single_msr": np.maximum(G2s0.p, 1 - G2s0.p).to_numpy(), "U1_ensemble_msr": np.maximum(G2.p, 1 - G2.p).to_numpy(), "U2_ensemble_var": -G2.p_var.to_numpy()}
    err_single = ((G2s0.p > 0.5).astype(int) != G2s0.y.astype(int)).to_numpy()
    mc_p, edl_rows = [], []
    for k in CFG.folds:
        Fd = load_feats("D1", k); std = Standardiser(subset(Fd, Fd["split"] == "train")); Fd = std(Fd); F_te = subset(Fd, Fd["split"] == "test")
        F_tr, F_va = split_train_val(subset(Fd, Fd["split"] == "train"), 0)
        te_t = _tensors(F_te, make_gate("G2", F_te, np.random.default_rng(0)), DEVICE)
        for s in CFG.seeds:                                                       # U3: stochastic passes over cached features
            m = DiagNet("G2").to(DEVICE); m.load_state_dict(torch.load(stage_dir(CFG.ds_diag) / f"model_diag_D1_G2_f{k}_s{s}.pt", map_location=DEVICE, weights_only=False)); m.train()
            with torch.no_grad():
                P = torch.stack([torch.sigmoid(m(*te_t[:4])[:, 0]) for _ in range(CFG.mc_passes)]).cpu().numpy()
            mc_p.append(pd.DataFrame(dict(id=F_te["id"], p_mc=P.mean(0), var_mc=P.var(0), seed=s)))
        r = train_diag("G2", F_tr, F_va, F_te, 0, CFG.diag_epochs, edl=True); S = r["test_out"].sum(1)      # U4: evidential head
        edl_rows.append(pd.DataFrame(dict(id=F_te["id"], p_edl=r["test_out"][:, 1] / S, conf_edl=1 - 2 / S)))
    mc = pd.concat(mc_p).groupby("id")[["p_mc", "var_mc"]].mean(); edl = pd.concat(edl_rows).set_index("id")
    unc["U3_mc_dropout"] = -mc.loc[G2.id, "var_mc"].to_numpy(); unc["U4_edl"] = edl.loc[G2.id, "conf_edl"].to_numpy()
    U = pd.DataFrame([dict(method=k, **selective_metrics(v, err_single if k == "U0_single_msr" else err.to_numpy(), CFG.coverage)) for k, v in unc.items()])[["method", "augrc", "aurc", "acc_at_cov"]]   # AUGRC primary, AURC secondary
    U.to_csv(out / "uncertainty.csv", index=False); print("\n", U.round(4).to_string(index=False))
    # ---- ASD head and negative transfer (multi-task G2 vs single-task ASD head on the same features) ------------------------
    asd_mt = ensemble(PI, "G2", "D1", task="asd")[0]; asd_st = []
    for k in CFG.folds:
        Fd = load_feats("D1", k); std = Standardiser(subset(Fd, Fd["split"] == "train")); Fd = std(Fd); F_te = subset(Fd, Fd["split"] == "test"); F_tr, F_va = split_train_val(subset(Fd, Fd["split"] == "train"), 0)
        r = train_diag("G2", F_tr, F_va, F_te, 0, CFG.diag_epochs, tasks=(1,)); asd_st.append(pd.DataFrame(dict(id=F_te["id"], y=F_te["y_asd"], p=SIGMOID(r["test_out"][:, 1]))))
    asd_st = pd.concat(asd_st); asd_st = asd_st[~np.isnan(asd_st.y)]
    summary["asd"] = dict(auroc_multitask=_safe_auc(roc_auc_score, asd_mt.y, asd_mt.p) if asd_mt is not None else np.nan, auroc_singletask=_safe_auc(roc_auc_score, asd_st.y, asd_st.p))
    summary["asd"]["negative_transfer"] = summary["asd"]["auroc_multitask"] - summary["asd"]["auroc_singletask"]
    # ---- error stratified by estimated and true quality; quality-estimator endpoints; gradient conflict ------------------------
    qv = []
    for k in CFG.folds:
        Fd = load_feats("D1", k); te_ = subset(Fd, Fd["split"] == "test"); qv.append(pd.DataFrame(dict(id=te_["id"], q_mean=te_["q"].mean(1), dice_mean=np.nanmean(te_["dice"], 1))))
    qv = pd.concat(qv).set_index("id").loc[G2.id]; strat = []
    for col in ("q_mean", "dice_mean"):
        bins = pd.qcut(qv[col].rank(method="first"), 4, labels=["Q1", "Q2", "Q3", "Q4"])
        for b in ["Q1", "Q2", "Q3", "Q4"]:
            s = (bins == b).to_numpy(); strat.append(dict(stratifier=col, quartile=b, n=int(s.sum()), abs_error=float(np.abs(G2.y[s] - G2.p[s]).mean()), auroc=_safe_auc(roc_auc_score, G2.y[s], G2.p[s])))
    strat = pd.DataFrame(strat); strat.to_csv(out / "error_by_quality.csv", index=False)
    c4 = seg_runs[seg_runs.arm.eq("C4") & seg_runs.init.eq("D1")]
    summary["quality_estimator"] = dict(spearman_rho=float(c4.quality_rho.mean()), mae=float(c4.quality_mae.mean()), **{f"rho_{n}": float(c4[f"rho_{n}"].mean()) for n in CHAMBERS},
                                        choice_per_fold=c4.quality_estimator_choice.tolist(), rho_test_hgb=float(c4.rho_test_hgb_morphology.mean()), rho_test_mlp=float(c4.rho_test_mlp_head.mean()),
                                        cpu_feature_ms_per_clip=float(c4.quality_feat_cpu_ms_per_clip.mean()), lv_right_of_rv_frac=float(c4.lv_right_of_rv_frac.mean()),
                                        note="training-video q_hat is mildly optimistic: the final core saw those videos' masks (accepted cost of one frozen deployable core)")
    summary["C4_dice"] = {n: float(c4[f"dice_{n}"].mean()) for n in CHAMBERS}; summary["C4_boundary_px"] = {n: float(c4[f"msd_px_{n}"].mean()) for n in CHAMBERS}
    if len(joint_runs):
        summary["G5"] = dict(rv_dice=float(joint_runs.dice_RV.mean()), auroc=float(joint_runs.auroc_pah.mean()), grad_cos_mean=float(joint_runs.grad_cos_mean.mean()))
    # ---- parameters, MACs, latency at batch 1 ------------------------------------------------------------------------------
    core = SegModel().to(DEVICE).eval(); head = DiagNet("G2").to(DEVICE).eval(); x = torch.randn(1, 2, CFG.frames, CFG.size, CFG.size, device=DEVICE)
    qe = pickle.load(open(stage_dir(CFG.ds_cores) / f"fold{CFG.folds[0]}" / "D1" / "quality_estimator.pkl", "rb")); qe_hgb = pickle.load(open(stage_dir(CFG.ds_cores) / f"fold{CFG.folds[0]}" / "D1" / "quality_estimator_hgb.pkl", "rb"))
    def sync():
        if DEVICE.type == "cuda": torch.cuda.synchronize()
    def one_clip():
        with torch.no_grad():
            t_c = time.perf_counter(); logits, f4, _ = core(x); prob = logits[-1].float().softmax(1); sync(); t_core = time.perf_counter() - t_c
            pr = prob.cpu().numpy(); pred = pr.argmax(1).astype(np.uint8); p_small = F.adaptive_avg_pool3d(prob, f4.shape[-3:])
            zt = (torch.einsum("bcthw,bdthw->btcd", p_small, f4.float()) / (p_small.sum(dim=(3, 4))[..., None].transpose(1, 2) + 1e-6))[0, :, 1:].cpu().numpy(); ft = f4.float().mean(dim=(3, 4))[0].T.cpu().numpy()
            t0 = time.perf_counter(); rows_, ffs, mfs = [], [], []
            for t in range(pred.shape[1]):
                for c in range(1, N_CLS):
                    rows_.append(dict(chamber=c, **chamber_frame_features(pr[0, :, t], pred[0, t], pred[0, t - 1] if t > 0 else None, pred[0, t + 1] if t + 1 < pred.shape[1] else None, c))); ffs.append(ft[t]); mfs.append(zt[t, c - 1])
            t_feat = time.perf_counter() - t0; rows_ = pd.DataFrame(rows_); t1 = time.perf_counter()
            q = torch.tensor(qe.predict(rows_, np.stack(ffs), np.stack(mfs)).reshape(pred.shape[1], 4).mean(0), device=DEVICE)[None]; t_est = time.perf_counter() - t1
            t2 = time.perf_counter(); head(f4.float().mean(dim=(2, 3, 4)), (torch.einsum("bcthw,bdthw->bcd", p_small, f4.float()) / (p_small.sum(dim=(2, 3, 4))[..., None] + 1e-6))[:, 1:], torch.zeros(1, 3, device=DEVICE), q); sync()
            return t_core, t_feat, t_est, time.perf_counter() - t2
    for _ in range(3): one_clip()
    ts = []
    for _ in range(10):
        t_start = time.perf_counter(); parts = one_clip(); ts.append((time.perf_counter() - t_start, *parts))
    ts = np.median(np.array(ts), 0)
    summary["efficiency"] = dict(params_core_decoder=count_params(core), params_diag=count_params(head), macs_core_decoder=count_macs(core, x), latency_ms_end_to_end_batch1=float(ts[0] * 1e3),
                                 latency_ms_core_decoder=float(ts[1] * 1e3), latency_ms_quality_features_cpu=float(ts[2] * 1e3), latency_ms_quality_estimator=float(ts[3] * 1e3), latency_ms_diag_head=float(ts[4] * 1e3),
                                 quality_estimator_used=type(qe).__name__, device=ENV_INFO["gpu"], input=list(x.shape), capacity=CFG.capacity)
    # ---- withdrawal conditions (reported as findings, never buried) ----------------------------------------------------------------
    T_ = pd.DataFrame(summary.get("confirmatory", []))
    def row(name): return T_[T_.comparison.str.startswith(name)].iloc[0] if len(T_) and T_.comparison.str.startswith(name).any() else None
    g21, g23, d10 = row("D1:G2 − D1:G1"), row("D1:G2 − D1:G3"), row("D1:G1 − D0:G1")
    findings["quality_gate_claim"] = dict(withdraw=bool(g21 is None or not (g21.seedfold_lo > 0) or (g23 is not None and not (g23.seedfold_lo > 0))),
                                          detail="withdraw if G2 does not beat G1 outside the seed interval or G3 matches G2",
                                          G2_minus_G1=None if g21 is None else [g21.seedfold_mean, g21.seedfold_lo, g21.seedfold_hi], G2_minus_G3=None if g23 is None else [g23.seedfold_mean, g23.seedfold_lo, g23.seedfold_hi])
    best = table.ens_auroc.max(); findings["parameter_budget_claim"] = dict(withdraw=bool(best <= 0.8932), best_ensemble_auroc=float(best), detail="CardiacNet reports 89.32 % AUC on an 8:1:1 split; ours is acquisition-grouped 5-fold CV, so the splits differ")
    g5 = summary.get("G5"); findings["freezing_claim"] = dict(withdraw=bool(g5 and g5["auroc"] >= table[table.arm.eq("G2") & table.init.eq("D1")].seed_mean_auroc.iloc[0] - (noise if np.isfinite(noise) else 0) and g5["rv_dice"] >= summary["C4_dice"]["RV"] - 0.02),
                                                             detail="withdraw if G5 matches G2 on AUROC with RV Dice within 0.02 of C4", G5=g5, C4_rv_dice=summary["C4_dice"]["RV"])
    c4d0 = seg_runs[seg_runs.arm.eq("C4") & seg_runs.init.eq("D0")]
    findings["H3_distillation"] = dict(supported=bool(d10 is not None and d10.seedfold_mean >= 0.02), D1_minus_D0_auroc_G1=None if d10 is None else float(d10.seedfold_mean),
                                       C4_rv_dice_D1=summary["C4_dice"]["RV"], C4_rv_dice_D0=(float(c4d0.dice_RV.mean()) if len(c4d0) else None), tested_on="G1 entropy-gate arm + C4 Dice (D0 has no quality estimator unless K1_D0_FULL=1)")
    a7 = next((s for s in screen if s["comparison"].startswith("D1:A0 − D1:A7")), None)
    findings["H4_frame_difference"] = dict(supported=bool(a7 and a7["mean"] > 0 and a7["exceeds_seed_noise"]), A0_minus_A7=a7)
    summary["findings"] = findings
    # ---- figures ---------------------------------------------------------------------------------------------------------------
    fig, ax = plt.subplots(1, 4, figsize=(20, 4.5))
    for (init, arm), E in ens.items():
        if init == "D1" and arm.startswith("G"):
            fpr, tpr, _ = roc_curve(E.y, E.p); ax[0].plot(fpr, tpr, label=f"{arm} AUROC {roc_auc_score(E.y, E.p):.3f}")
    ax[0].plot([0, 1], [0, 1], "k:"); ax[0].set_title("ROC, seed ensembles"); ax[0].legend(fontsize=8)
    for method in ("none", "temperature", "platt"):
        d = oof(PI, "G2", "D1", 0) if method == "none" else calibrated_oof(PI, "G2", "D1", 0, method); o = np.argsort(d.p); bins = np.array_split(o, 8)
        ax[1].plot([d.p.to_numpy()[b].mean() for b in bins], [d.y.to_numpy()[b].mean() for b in bins], "o-", label=method)
    ax[1].plot([0, 1], [0, 1], "k:"); ax[1].set_title("Reliability (G2, seed 0)"); ax[1].legend(fontsize=8)
    for k_, v in unc.items():
        o = np.argsort(-v); e = err.to_numpy()[o]; ax[2].plot(np.arange(1, len(e) + 1) / len(e), np.cumsum(e) / len(e), label=k_)
    ax[2].set_title("Generalised risk–coverage"); ax[2].set_xlabel("coverage"); ax[2].legend(fontsize=8)
    if len(joint_runs):
        for _, r in joint_runs.iterrows(): ax[3].plot(json.loads(r.grad_cos_curve), alpha=0.6)
        ax[3].axhline(0, color="k", ls=":"); ax[3].set_title("G5 task-gradient cosine per epoch")
    plt.tight_layout(); plt.savefig(out / "k1_figures.png", dpi=110); plt.close()
    (out / "summary.json").write_text(json.dumps(summary, indent=1, default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else str(o)))
    write_version(CFG.ds_final)
    print("\nFindings:"); [print(f"  {k}: {json.dumps(v, default=str)[:300]}") for k, v in findings.items()]
    print(f"\n[stage 7] summary → {out/'summary.json'} | efficiency: {summary['efficiency']}")


if 7 in CFG.stages:
    stage7_statistics()

[stage 7] depends on: ['cardiacnet-cache-v1', 'diag-features-v1', 'diag-results-v1', 'seg-cores-v1']
   cardiacnet-cache-v1    -> k1_work/cardiacnet-cache-v1  (2026-09-29T13:37:36)
   diag-features-v1       -> k1_work/diag-features-v1  (2026-09-29T13:43:26)
   diag-results-v1        -> k1_work/diag-results-v1  (2026-09-29T13:43:33)
   seg-cores-v1           -> k1_work/seg-cores-v1  (2026-09-29T13:43:26)


   init arm  n_seeds  ens_auroc  ens_auprc  ens_brier  ens_nll  ens_ece  ens_cal_slope  ens_acc  ens_n  seed_mean_auroc  n_true_oracle
     D1  G0        3      0.893      0.953      0.137    0.446    0.268          1.683    0.895     19            0.817            NaN
     D1  G1        3      0.857      0.943      0.130    0.425    0.255          1.330    0.842     19            0.802            NaN
     D1  G2        3      0.845      0.940      0.137    0.441    0.254          1.175    0.842     19            0.821            NaN
     D1  G3        3      0.845      0.940      0.137    0.442    0.257          1.205    0.842     19            0.821            NaN
     D1  G4        3      0.857      0.943      0.146    0.468    0.274          1.166    0.842     19            0.821           26.0
     D1  G5        3      0.440      0.689      0.243    0.677    0.375          0.143    0.526     19            0.472            NaN
     D1  A0        1      0.417      0.664      0.262  


    comparison  boot_delta  boot_ci_lo  boot_ci_hi  boot_n_groups  perm_p  seedfold_mean  seedfold_lo  seedfold_hi  seedfold_n  perm_p_holm
D1:G2 − D1:G1     -0.0119     -0.0445      0.0000             12  0.7562         0.0301      -0.1122       0.1724           6       1.0000
D1:G2 − D1:G3      0.0000     -0.0000      0.0000             12  1.0000         0.0000       0.0000       0.0000           6       1.0000
D1:G2 − D1:G0     -0.0476     -0.1333      0.0000             12  0.2139        -0.0370      -0.3381       0.2640           6       0.8557
D1:G1 − D0:G1      0.0119     -0.1442      0.2403             12  0.9552        -0.0069      -0.1194       0.1055           6       1.0000

      method    nll    ece  brier  cal_slope  auroc
       none 0.5976 0.2287 0.1736     0.4331 0.8214
temperature 0.9111 0.2446 0.1719     0.6436 0.7976
      platt 0.9396 0.2420 0.1904     0.4576 0.7976



          method  augrc   aurc  acc_at_cov
  U0_single_msr 0.1524 0.2642      0.6111
U1_ensemble_msr 0.0706 0.1390      0.8889
U2_ensemble_var 0.0873 0.1596      0.8333
  U3_mc_dropout 0.0817 0.1554      0.8889
         U4_edl 0.0540 0.0787      0.8333



Findings:
  quality_gate_claim: {"withdraw": true, "detail": "withdraw if G2 does not beat G1 outside the seed interval or G3 matches G2", "G2_minus_G1": [0.030092592592592577, -0.11224565487797551, 0.17243084006316067], "G2_minus_G3": [0.0, 0.0, 0.0]}
  parameter_budget_claim: {"withdraw": true, "best_ensemble_auroc": 0.8928571428571429, "detail": "CardiacNet reports 89.32 % AUC on an 8:1:1 split; ours is acquisition-grouped 5-fold CV, so the splits differ"}
  freezing_claim: {"withdraw": false, "detail": "withdraw if G5 matches G2 on AUROC with RV Dice within 0.02 of C4", "G5": {"rv_dice": 0.5934098349771402, "auroc": 0.5231481481481481, "grad_cos_mean": 0.01299842860995945}, "C4_rv_dice": 0.867777693192615}
  H3_distillation: {"supported": false, "D1_minus_D0_auroc_G1": -0.0069444444444444475, "C4_rv_dice_D1": 0.867777693192615, "C4_rv_dice_D0": 0.5889626101215693, "tested_on": "G1 entropy-gate arm + C4 Dice (D0 has no quality estimator unless K1_D0_FULL=1)"}
  H4_frame_difference: